<a href="https://colab.research.google.com/github/crystalloide/IA_102_IA_Agentique/blob/main/IA102_Atelier1_Human_in_the_Loop_LangGraph_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atelier IA102 - Human-in-the-Loop : supervision et validation humaine

## Construction progressive d'un chatbot Human-in-the-Loop avec LangGraph

**Formation** : IA102 — IA Agentique : conception d'applications multi-agents basés sur des LLMs (LLM-MA)
*Réf. IA102 · 2 jours (14 h) · présentiel et/ou classe à distance*

**Module 1 — Fondamentaux et modélisation des architectures multi-agents · Atelier de fin de module**
**Durée en salle : 1 h 15** (parcours guidé, parties 0 à 10) · extensions en autonomie : parties 11 à 13
**Date de validation technique du notebook : 20 septembre 2026**

---

### Rattachement au programme

Cet atelier clôt le module 1 et met en pratique la dernière ligne de son programme —
*« Intégration du concept Human-in-the-Loop pour assurer supervision et validation humaine »* —
en réinvestissant les trois points enseignés juste avant :

| Point du programme (module 1) | Où il est mis en pratique |
|---|---|
| Étude de l'approche ReACT et identification de ses **limites structurelles** | parties 6 et 8 : la boucle ReACT agit sans demander ; le HITL est la réponse à cette limite |
| Modélisation en **graphes**, gestion des états et contrôle de la récursivité | parties 6 et 7 : `StateGraph`, réducteurs, `recursion_limit` |
| Architectures de **mémoire court terme et long terme** | partie 5 : `SqliteSaver` et `SqliteStore` |

Il prépare également le **module 5** (gouvernance, AI Act, explicabilité) : la partie 10 produit
la piste d'audit qu'exige une supervision humaine opposable.

### Objectif pédagogique

À l'issue de cet atelier, vous saurez construire un agent conversationnel qui **ne peut pas agir
seul** sur le monde extérieur : toute action irréversible est suspendue, présentée à un opérateur
humain, puis exécutée, corrigée ou abandonnée selon sa décision.

Vous développerez, brique par brique, un agent intégrant :

| Brique | Mise en œuvre |
|---|---|
| **Mémoire persistante** | `SqliteSaver` (mémoire courte, par conversation) + `SqliteStore` (mémoire longue, par utilisateur) |
| **Outils de recherche web** | Outil `recherche_web` en cascade : Tavily → DuckDuckGo → corpus local hors-ligne |
| **Validation humaine dans le flux** | `interrupt()` / `Command(resume=...)` — les 4 patrons : approuver, modifier, refuser, consigner |
| **Implémentation LangGraph** | `StateGraph` explicite, puis variante `create_agent` + `HumanInTheLoopMiddleware` |

### Progression de l'atelier

| Partie | Contenu | Durée |
|---|---|---|
| 0 | Mise en place de l'environnement (versions figées) | 5 min |
| 1 | Génération des fichiers de test | 3 min |
| 2 | Configuration des clés API — **facultative** | 2 min |
| 3 | Le modèle de langage (réel ou simulé) | 5 min |
| 4 | Les outils, classés lecture seule / effet de bord | 10 min |
| 5 | Les deux mémoires | 10 min |
| 6 | L'état, les nœuds, et le point d'arrêt humain | 15 min |
| 7 | Compilation et lecture du graphe | 5 min |
| 8 | 6 scénarios guidés de validation humaine | 15 min |
| 9 | Persistance : reprise après redémarrage | 5 min |
| 10 | Journal d'audit | 5 min |
| | **Total du parcours guidé** | **1 h 20** |
| 11 | Mode interactif libre | *en autonomie* |
| 12 | Variante industrielle : `HumanInTheLoopMiddleware` | *en autonomie* |
| 13 | Exercices et corrigés | *en autonomie* |

> **Note pour le formateur** : si le temps manque, les parties 4 et 5 peuvent être exécutées sans
> commentaire (le code y est lu, pas écrit). Les parties **6 et 8 sont le cœur de l'atelier** et ne
> doivent pas être comprimées. Les parties 11 à 13 sont conçues pour être reprises par le stagiaire
> après la session.

### Prérequis

Ceux de la formation : cours **BI108** ou expérience préalable sur les LLMs ; la maîtrise d'un
langage de script type Python est recommandée. Pour cet atelier en particulier :

- Un compte Google (Colab) ou un Jupyter local en Python 3.10+.
- **Aucune clé API n'est nécessaire.** Le notebook embarque un modèle simulé déterministe et un
  corpus documentaire local : il s'exécute intégralement hors-ligne. Si vous disposez d'une clé
  OpenAI, Anthropic ou Tavily, la partie 2 permet de les activer — le reste du notebook est
  rigoureusement identique.
- **Aux chefs de projet IA** : aucune ligne de code n'est à écrire dans le parcours guidé. Les
  cellules s'exécutent telles quelles ; ce qui compte est de savoir lire la trace d'exécution et
  de reconnaître le moment où l'agent s'arrête.

> **Mode d'emploi** : exécutez les cellules **dans l'ordre**, de haut en bas
> (`Exécution ▸ Tout exécuter` fonctionne du premier coup).

---
## Partie 0 — Mise en place de l'environnement

Les versions sont **figées** (`==`) volontairement. L'écosystème LangChain/LangGraph évolue vite ;
un atelier reproductible doit épingler ses dépendances. Les versions ci-dessous sont celles
validées au **20 septembre 2026**.

Points d'attention pour cet atelier :

- `langgraph 1.x` : `interrupt()` et `Command` sont dans `langgraph.types`.
- `langchain-core 1.x` : le contenu d'un message peut être une chaîne **ou** une liste de blocs
  (selon le fournisseur) — d'où la fonction utilitaire `texte_de()` plus bas.
- `langgraph-checkpoint-sqlite` fournit **à la fois** `SqliteSaver` (mémoire courte) et
  `SqliteStore` (mémoire longue).

L'installation prend environ 1 minute.

> **Deux messages sans gravité peuvent apparaître sur Colab :**
> `Note: you may need to restart the kernel…` et un avertissement du résolveur de dépendances
> concernant des paquets préinstallés que cet atelier n'utilise pas. **Ignorez-les** : la cellule
> de vérification suivante fait foi. Ce n'est que si elle signale un import impossible qu'il faut
> faire `Exécution ▸ Redémarrer la session`, puis relancer ces deux cellules.

In [ ]:
%pip install -q \
    "langgraph==1.2.11" \
    "langchain==1.4.2" \
    "langchain-core==1.6.3" \
    "langgraph-checkpoint-sqlite==3.1.1" \
    "langchain-openai==1.6.2" \
    "langchain-anthropic==1.7.2" \
    "ddgs==9.16.0" \
    "grandalf==0.8"

print("\nInstallation terminée.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 431.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 5.1 MB/s eta 0:00:00

Installation terminée.


In [ ]:
# --- Vérification de l'environnement : cette cellule doit afficher « ENVIRONNEMENT PRÊT » ---
import platform
from importlib.metadata import version as version_installee

ATTENDU = {
    "langgraph": "1.2.11",
    "langchain": "1.4.2",
    "langchain-core": "1.6.3",
    "langgraph-checkpoint-sqlite": "3.1.1",
}

print(f"Python : {platform.python_version()}")
probleme = False
for paquet, version_attendue in ATTENDU.items():
    try:
        v = version_installee(paquet)
        etat = "ok" if v == version_attendue else f"ATTENTION (attendu {version_attendue})"
        print(f"  {paquet:<28} {v:<10} {etat}")
    except Exception as e:                                   # pragma: no cover
        probleme = True
        print(f"  {paquet:<28} NON INSTALLÉ : {e}")

# Les imports réellement utilisés dans la suite de l'atelier :
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore
from langchain_core.tools import tool
from langchain_core.messages import (AIMessage, BaseMessage, HumanMessage,
                                     SystemMessage, ToolMessage)

print("\nENVIRONNEMENT PRÊT." if not probleme else
      "\nProblème détecté : menu « Exécution ▸ Redémarrer la session », puis relancez ces 2 cellules.")

Python : 3.13.15
  langgraph                    1.2.11     ok
  langchain                    1.4.2      ok
  langchain-core               1.6.3      ok
  langgraph-checkpoint-sqlite  3.1.1      ok

ENVIRONNEMENT PRÊT.


---
## Partie 1 — Génération des fichiers de test

L'atelier ne dépend d'**aucune ressource externe**. Cette cellule crée tout ce dont l'agent aura
besoin :

| Dossier | Rôle |
|---|---|
| `corpus/` | 5 fiches documentaires — c'est le **moteur de recherche hors-ligne** de secours |
| `notes/` | 2 notes internes d'équipe, lues par l'outil `lire_note` |
| `documents/` | 3 documents, dont un contrat à **ne pas** supprimer — support du scénario de refus |
| `corbeille/` | destination des suppressions validées |
| `sortie/` | `boite_envoi.jsonl` (e-mails simulés) et `journal_validations.jsonl` (piste d'audit) |
| `memoire/` | les deux bases SQLite de l'agent |

> **Pourquoi un corpus local ?** Un atelier doit fonctionner même sans réseau, derrière un proxy
> d'entreprise, ou quand un moteur de recherche gratuit limite le débit. L'outil `recherche_web`
> essaiera d'abord Tavily, puis DuckDuckGo, et retombera **silencieusement** sur ce corpus.

In [ ]:
from pathlib import Path

# /content existe sur Colab ; sinon on travaille dans le dossier courant.
RACINE = Path("/content/atelier_hitl") if Path("/content").exists() else Path("./atelier_hitl")

DOSSIERS = {
    "corpus":    RACINE / "corpus",
    "notes":     RACINE / "notes",
    "documents": RACINE / "documents",
    "corbeille": RACINE / "corbeille",
    "sortie":    RACINE / "sortie",
    "memoire":   RACINE / "memoire",
}

CORPUS = {
    "langgraph.md": """# LangGraph : le framework de graphes d'agents
LangGraph (version 1.x, 2026) modélise une application LLM comme un graphe d'états.
Les briques : StateGraph (définition), nœuds (fonctions Python), arêtes (transitions),
checkpointer (persistance de l'état à chaque super-étape).
Le checkpointer est ce qui rend possible la reprise d'exécution : sans lui, pas de
Human-in-the-Loop, pas de mémoire conversationnelle, pas de time-travel.
Mots-clés : StateGraph, MessagesState, add_messages, compile, thread_id.
""",
    "human_in_the_loop.md": """# Human-in-the-Loop (HITL)
Le Human-in-the-Loop insère un point d'arrêt contrôlé dans l'exécution d'un agent
autonome, afin qu'un opérateur humain valide, corrige ou refuse une action avant
qu'elle ne produise un effet irréversible (envoi d'e-mail, suppression, paiement).
Les quatre patrons canoniques :
1. Approve / Reject : autoriser ou bloquer l'appel d'outil.
2. Edit : corriger les arguments proposés par le modèle avant exécution.
3. Review & Feedback : renvoyer un commentaire au modèle sans exécuter l'outil.
4. Input Validation : demander une information manquante à l'humain.
En LangGraph, la primitive est la fonction interrupt(), et la reprise se fait avec
Command(resume=...). L'état est figé dans le checkpointer pendant la pause.
""",
    "memoire_agents.md": """# Mémoire des agents conversationnels
On distingue deux mémoires. La mémoire courte (short-term) est l'historique de la
conversation courante : en LangGraph elle est portée par le checkpointer et
identifiée par un thread_id. La mémoire longue (long-term) est un savoir durable
partagé entre conversations : en LangGraph elle est portée par un Store,
organisé en namespaces, par exemple ("memoire", user_id).
Bonne pratique : ne jamais stocker en mémoire longue des données sensibles
(identifiants, données de santé, coordonnées bancaires).
""",
    "rgpd_validation.md": """# Validation humaine et conformité
Le règlement européen sur l'IA (AI Act) et le RGPD (article 22) encadrent les
décisions entièrement automatisées. Un contrôle humain significatif suppose que
l'opérateur dispose de l'information utile, du temps nécessaire, et d'un pouvoir
réel d'interrompre. Un bouton « OK » sans contexte ne constitue pas une supervision
humaine effective : c'est le rubber-stamping, un anti-patron documenté.
Traçabilité : journaliser qui a validé, quoi, et quand.
""",
    "outils_agents.md": """# Outils (tools) et appels de fonctions
Un outil est une fonction Python exposée au modèle avec un schéma d'arguments.
Le modèle n'exécute rien : il émet un tool_call (nom + arguments JSON). C'est
l'orchestrateur qui exécute. Cette séparation est précisément le point d'insertion
du contrôle humain : entre la proposition du modèle et l'exécution effective.
Classification recommandée : outils en lecture seule (exécution directe) et outils
à effet de bord (validation humaine obligatoire).
""",
}

NOTES = {
    "reunion_2026_09_15.md": """# Réunion projet Atlas — 15 septembre 2026
Participants : Stéphane, Leïla, Marc.
Décisions : livraison du prototype HITL pour le 30 septembre.
Point ouvert : choix du checkpointer (SQLite en local, Postgres en production).
""",
    "checklist_mise_en_production.md": """# Checklist mise en production d'un agent
1. Tous les outils à effet de bord passent par une validation humaine.
2. Le checkpointer est persistant (pas InMemorySaver).
3. Les identifiants sont lus depuis des variables d'environnement.
4. Chaque validation est journalisée (qui, quoi, quand).
5. Un délai d'expiration est prévu si l'humain ne répond pas.
""",
}

DOCUMENTS = {
    "brouillon_obsolete.txt": "Ancien brouillon de spécification, remplacé le 12/09/2026.\n",
    "export_temporaire.txt": "Export temporaire généré automatiquement. Peut être supprimé.\n",
    "contrat_client_signe.txt": "CONTRAT SIGNÉ — NE PAS SUPPRIMER — référence AT-2026-118.\n",
}


def generer_fichiers_de_test(verbeux: bool = True) -> Path:
    for dossier in DOSSIERS.values():
        dossier.mkdir(parents=True, exist_ok=True)
    for nom, contenu in CORPUS.items():
        (DOSSIERS["corpus"] / nom).write_text(contenu, encoding="utf-8")
    for nom, contenu in NOTES.items():
        (DOSSIERS["notes"] / nom).write_text(contenu, encoding="utf-8")
    for nom, contenu in DOCUMENTS.items():
        (DOSSIERS["documents"] / nom).write_text(contenu, encoding="utf-8")
    # fichiers de sortie remis à zéro à chaque exécution de l'atelier
    (DOSSIERS["sortie"] / "boite_envoi.jsonl").write_text("", encoding="utf-8")
    (DOSSIERS["sortie"] / "journal_validations.jsonl").write_text("", encoding="utf-8")
    if verbeux:
        print(f"Racine        : {RACINE.resolve()}")
        print(f"Corpus web    : {len(CORPUS)} fiches")
        print(f"Notes         : {len(NOTES)} notes")
        print(f"Documents     : {len(DOCUMENTS)} documents")
        print(f"Sorties       : boite_envoi.jsonl, journal_validations.jsonl (remis à zéro)")
        print(f"TOTAL         : {len(CORPUS) + len(NOTES) + len(DOCUMENTS)} fichiers de test générés.")
    return None


generer_fichiers_de_test()

Racine        : /content/atelier_hitl
Corpus web    : 5 fiches
Notes         : 2 notes
Documents     : 3 documents
Sorties       : boite_envoi.jsonl, journal_validations.jsonl (remis à zéro)
TOTAL         : 10 fichiers de test générés.


In [ ]:
# Arborescence générée
for dossier in sorted(RACINE.rglob("*")):
    if dossier.is_dir():
        print(f"{dossier.relative_to(RACINE)}/")
        for f in sorted(dossier.iterdir()):
            print(f"    {f.name:<36} {f.stat().st_size:>6} octets")

corbeille/
corpus/
    human_in_the_loop.md                    761 octets
    langgraph.md                            517 octets
    memoire_agents.md                       554 octets
    outils_agents.md                        507 octets
    rgpd_validation.md                      502 octets
documents/
    brouillon_obsolete.txt                   61 octets
    contrat_client_signe.txt                 65 octets
    export_temporaire.txt                    67 octets
memoire/
notes/
    checklist_mise_en_production.md         359 octets
    reunion_2026_09_15.md                   229 octets
sortie/
    boite_envoi.jsonl                         0 octets
    journal_validations.jsonl                 0 octets


---
## Partie 2 — Clés API (facultatif)

**Cette cellule est optionnelle.** Laissez `UTILISER_CLES_API = False` pour exécuter l'atelier
hors-ligne avec le modèle simulé : tous les scénarios fonctionnent à l'identique.

Passez à `True` si vous voulez brancher un vrai LLM. Une invite sécurisée (`getpass`) vous
demandera alors les clés dont vous disposez ; laissez vide celles que vous n'avez pas.

| Variable | Effet |
|---|---|
| `OPENAI_API_KEY` | l'agent utilise un modèle OpenAI |
| `ANTHROPIC_API_KEY` | à défaut d'OpenAI, l'agent utilise un modèle Anthropic |
| `TAVILY_API_KEY` | l'outil `recherche_web` interroge l'API Tavily en priorité |

> **Bonne pratique** : jamais de clé en clair dans une cellule. Sur Colab, préférez même le
> gestionnaire de secrets (icône 🔑 dans la barre latérale) :
> `from google.colab import userdata ; userdata.get('OPENAI_API_KEY')`.

In [ ]:
import os

UTILISER_CLES_API = False        # <<< passez à True pour saisir vos clés

if UTILISER_CLES_API:
    from getpass import getpass
    for nom in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
        valeur = getpass(f"{nom} (laisser vide pour ignorer) : ").strip()
        if valeur:
            os.environ[nom] = valeur

presentes = [n for n in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY") if os.environ.get(n)]
print("Clés actives :", ", ".join(presentes) if presentes else "aucune → mode hors-ligne intégral")

Clés actives : aucune → mode hors-ligne intégral


---
## Partie 3 — Le modèle de langage

Un atelier qui dépend d'une clé API est un atelier qui ne fonctionne pas le jour de la formation.
Nous adoptons donc une **stratégie de repli** :

```
clé OpenAI ?  ──oui──▶ ChatOpenAI
     │non
clé Anthropic ? ─oui─▶ ChatAnthropic
     │non
     └──────────────▶ ModeleSimule   (déterministe, local, sans réseau)
```

`ModeleSimule` est une sous-classe de `BaseChatModel`. Elle reproduit le seul comportement qui
nous intéresse ici : **soit le modèle émet un `tool_call`, soit il rédige une réponse**. Ses règles
sont volontairement lisibles — l'objet de l'atelier est le contrôle humain, pas la qualité de
génération.

> **Point clé à retenir** : un LLM **n'exécute jamais** un outil. Il produit un objet
> `tool_call = {"name": ..., "args": {...}, "id": ...}`. C'est l'orchestrateur (LangGraph) qui
> exécute. **C'est exactement dans cet interstice que s'insère la validation humaine.**

In [ ]:
from __future__ import annotations

import re
import unicodedata
import uuid
from typing import Any, Optional

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import (AIMessage, BaseMessage, HumanMessage,
                                     SystemMessage, ToolMessage)
from langchain_core.outputs import ChatGeneration, ChatResult


# ---------------------------------------------------------------- utilitaires
def sans_accents(texte: str) -> str:
    """Minuscules sans accents : sert à comparer du texte français de façon robuste.

    La transformation se fait caractère par caractère afin de **préserver la longueur**
    de la chaîne : les positions trouvées par une expression régulière sur le résultat
    restent donc valides sur le texte d'origine (utilisé en partie 5).
    """
    sortie = []
    for caractere in texte:
        base = unicodedata.normalize("NFD", caractere)
        sortie.append(base[0] if base else caractere)
    return "".join(sortie).lower()


def texte_de(message: BaseMessage) -> str:
    """Extrait le texte d'un message, que son contenu soit une chaîne ou une liste de blocs.

    Indispensable avec langchain-core 1.x : selon le fournisseur, `message.content` est
    soit `"bonjour"`, soit `[{"type": "text", "text": "bonjour"}, ...]`.
    """
    contenu = message.content
    if isinstance(contenu, str):
        return contenu
    morceaux = []
    for bloc in contenu or []:
        if isinstance(bloc, str):
            morceaux.append(bloc)
        elif isinstance(bloc, dict) and bloc.get("type") == "text":
            morceaux.append(bloc.get("text", ""))
    return "".join(morceaux)


# ------------------------------------------------------------- modèle simulé
class ModeleSimule(BaseChatModel):
    """Modèle déterministe local : aucune clé API, aucun appel réseau.

    Il décide, à partir de règles explicites, s'il faut appeler un outil ou répondre.
    Les attributs sont déclarés en champs Pydantic (BaseChatModel est un modèle Pydantic v2).
    """

    outils_disponibles: list[str] = []
    max_appels_par_tour: int = 2

    @property
    def _llm_type(self) -> str:
        return "modele-simule-atelier-hitl"

    def bind_tools(self, tools: list[Any], **kwargs: Any) -> "ModeleSimule":
        """Équivalent local de `ChatOpenAI.bind_tools` : mémorise les outils exposés."""
        noms = [getattr(t, "name", None) or getattr(t, "__name__", str(t)) for t in tools]
        return self.__class__(outils_disponibles=noms,
                              max_appels_par_tour=self.max_appels_par_tour)

    # -------- fabrication d'un appel d'outil, au format attendu par LangChain
    @staticmethod
    def _appel(nom: str, args: dict) -> dict:
        return {"name": nom, "args": args,
                "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}

    # -------- lecture du contexte conversationnel
    @staticmethod
    def _dernier_humain(messages: list[BaseMessage]) -> str:
        for m in reversed(messages):
            if isinstance(m, HumanMessage):
                return texte_de(m)
        return ""

    @staticmethod
    def _memoire(messages: list[BaseMessage]) -> str:
        for m in messages:
            if isinstance(m, SystemMessage) and "Faits mémorisés" in texte_de(m):
                return texte_de(m).split("Faits mémorisés", 1)[1]
        return ""

    @staticmethod
    def _appels_du_tour(messages: list[BaseMessage]) -> list[str]:
        """Noms des outils déjà appelés depuis le dernier message de l'utilisateur."""
        noms = []
        for m in reversed(messages):
            if isinstance(m, HumanMessage):
                break
            if isinstance(m, AIMessage) and m.tool_calls:
                noms.extend(a["name"] for a in m.tool_calls)
        return noms

    # -------- extraction d'arguments
    @staticmethod
    def _extraire_email(texte: str) -> str:
        trouve = re.search(r"[\w\.\-\+]+@[\w\.\-]+\.\w+", texte)
        return trouve.group(0) if trouve else "destinataire@exemple.fr"

    @staticmethod
    def _extraire_sujet_recherche(texte: str) -> str:
        t = texte.strip().rstrip("?.!")
        for amorce in ["peux-tu chercher des informations sur", "cherche des informations sur",
                       "fais une recherche sur", "cherche sur le web", "recherche sur le web",
                       "documente-toi sur", "documente toi sur", "peux-tu m'expliquer",
                       "peux tu m'expliquer", "explique-moi", "explique moi",
                       "qu'est-ce que", "qu'est ce que", "c'est quoi",
                       "recherche", "cherche", "explique", "définis"]:
            if sans_accents(t).startswith(sans_accents(amorce)):
                t = t[len(amorce):].strip(" :,'’")
                break
        for separateur in [" puis ", " ensuite ", " et envoie", " et écris", " et supprime"]:
            position = sans_accents(t).find(sans_accents(separateur))
            if position > 0:
                t = t[:position]
                break
        return t.strip(" :,'’") or "human in the loop"

    @classmethod
    def _extraire_sujet_email(cls, texte: str) -> str:
        """Isole le thème du message, en retirant la consigne d'envoi et l'adresse."""
        t = texte.strip().rstrip("?.!")
        for marqueur in [" au sujet du ", " au sujet de la ", " au sujet des ",
                         " au sujet de l'", " au sujet de ",
                         " à propos du ", " à propos de la ", " à propos des ",
                         " à propos de l'", " à propos de ",
                         " concernant le ", " concernant la ", " concernant les ",
                         " concernant l'", " concernant ",
                         " sur le ", " sur la ", " sur les ", " sur l'", " sur "]:
            position = sans_accents(t).find(sans_accents(marqueur))
            if position > 0:
                return t[position + len(marqueur):].strip(" :,'’")[:70] or "votre demande"
        sujet = cls._extraire_sujet_recherche(t)
        sujet = re.sub(r"(envoie|envoyer|écris|écrire|rédige|rédiger)\s+(un\s+)?(e-?mail|mail|courriel)\s*(à\s+)?",
                       "", sujet, flags=re.I)
        sujet = re.sub(r"[\w\.\-\+]+@[\w\.\-]+\.\w+", "", sujet).strip(" :,-'’")
        return sujet[:70] or "votre demande"

    # ======================= cœur du modèle : décider =========================
    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        dernier = messages[-1]
        demande = self._dernier_humain(messages)
        d = sans_accents(demande)
        deja = self._appels_du_tour(messages)

        besoin_recherche = any(k in d for k in [
            "cherche", "recherche", "web", "internet", "actualite", "qu'est ce que",
            "qu'est-ce que", "c'est quoi", "explique", "documentation", "definis",
            "definition", "comment fonctionne",
        ])

        # --- 1. Un outil vient de répondre : synthétiser, ou enchaîner un 2e outil.
        if isinstance(dernier, ToolMessage):
            resultats = [m for m in messages if isinstance(m, ToolMessage)]
            enchainer_mail = (
                any(k in d for k in ["mail", "e-mail", "email", "envoie", "envoyer"])
                and "envoyer_email" in self.outils_disponibles
                and "envoyer_email" not in deja
                and len(deja) < self.max_appels_par_tour
            )
            if enchainer_mail:
                return self._resultat(
                    "Recherche terminée. Je prépare l'e-mail de synthèse : il attend votre validation.",
                    [self._appel("envoyer_email", {
                        "destinataire": self._extraire_email(demande),
                        "sujet": f"Synthèse : {self._extraire_sujet_recherche(demande)[:60]}",
                        "corps": "Bonjour,\n\nVoici la synthèse demandée.\n\n"
                                 + texte_de(resultats[-1])[:600]
                                 + "\n\nBien cordialement,\nVotre agent.",
                    })])
            return self._resultat(self._synthese(demande, resultats))

        # --- 2. Suppression de document (OUTIL SENSIBLE)
        if any(k in d for k in ["supprime", "supprimer", "efface", "effacer"]) \
                and "supprimer_document" in self.outils_disponibles \
                and "supprimer_document" not in deja:
            trouve = re.search(r"([\w\-\.]+\.(?:txt|md|csv|pdf))", demande)
            cible = trouve.group(1) if trouve else "brouillon_obsolete.txt"
            return self._resultat(
                f"Je propose de supprimer « {cible} ». Cette action demande votre validation.",
                [self._appel("supprimer_document", {"nom_fichier": cible})])

        # --- 3. Envoi d'e-mail (OUTIL SENSIBLE)
        #     si l'utilisateur demande AUSSI une recherche, on cherche d'abord (point 5)
        if any(k in d for k in ["mail", "e-mail", "email"]) and not besoin_recherche \
                and "envoyer_email" in self.outils_disponibles \
                and "envoyer_email" not in deja:
            sujet = self._extraire_sujet_email(demande)
            return self._resultat(
                "J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.",
                [self._appel("envoyer_email", {
                    "destinataire": self._extraire_email(demande),
                    "sujet": sujet[:1].upper() + sujet[1:],
                    "corps": f"Bonjour,\n\nFaisant suite à votre demande concernant {sujet}, "
                             f"voici le message préparé automatiquement par l'agent.\n\n"
                             f"Bien cordialement,\nVotre agent.",
                })])

        # --- 4. Mémorisation explicite
        if any(k in d for k in ["retiens", "memorise", "souviens", "note que", "enregistre que"]) \
                and "enregistrer_preference" in self.outils_disponibles \
                and "enregistrer_preference" not in deja:
            valeur = re.sub(r"^\s*(retiens|mémorise|souviens[- ]toi|note|enregistre)\s*(que)?\s*:?\s*",
                            "", demande, flags=re.I).strip()
            return self._resultat("Je note cette information en mémoire longue.",
                                  [self._appel("enregistrer_preference",
                                               {"cle": "note_utilisateur", "valeur": valeur})])

        # --- 5. Lecture d'une note interne
        if "note" in d and any(k in d for k in ["lis", "lire", "ouvre", "affiche", "reunion", "checklist"]) \
                and "lire_note" in self.outils_disponibles and "lire_note" not in deja:
            return self._resultat("Je consulte les notes internes.",
                                  [self._appel("lire_note",
                                               {"sujet": self._extraire_sujet_recherche(demande)})])

        # --- 6. Recherche documentaire
        if besoin_recherche and "recherche_web" in self.outils_disponibles \
                and "recherche_web" not in deja:
            return self._resultat("Je lance une recherche documentaire.",
                                  [self._appel("recherche_web",
                                               {"requete": self._extraire_sujet_recherche(demande)})])

        # --- 7. Réponse directe, enrichie par la mémoire longue
        return self._resultat(self._reponse_directe(demande, self._memoire(messages)))

    # ======================= rédaction des réponses ===========================
    def _resultat(self, texte: str, appels: Optional[list[dict]] = None) -> ChatResult:
        message = AIMessage(content=texte, tool_calls=appels or [])
        return ChatResult(generations=[ChatGeneration(message=message)])

    def _synthese(self, demande: str, resultats: list[ToolMessage]) -> str:
        dernier = resultats[-1]
        corps = texte_de(dernier).strip()
        nom = getattr(dernier, "name", "") or ""

        # a) l'opérateur humain a bloqué l'action
        if corps.startswith("Action REFUSÉE"):
            motif = corps.split("Motif :", 1)[-1].split(".")[0].strip() if "Motif :" in corps else "non précisé"
            return (f"L'action n'a pas été effectuée : elle a été refusée lors de la validation "
                    f"humaine (motif : {motif}). Je ne la relance pas. Dites-moi comment "
                    f"vous souhaitez procéder.")
        if corps.startswith("Action non exécutée"):
            consigne = corps.split("opérateur :", 1)[-1].strip()
            return (f"Bien noté, je n'ai rien envoyé. Consigne retenue : {consigne} "
                    f"Je peux préparer une nouvelle version sur cette base.")

        # b) outils à effet de bord effectivement exécutés
        if nom in ("envoyer_email", "supprimer_document"):
            return "C'est fait : " + corps
        if nom == "enregistrer_preference":
            return corps + " Je m'en souviendrai lors de nos prochaines conversations."

        # c) outils de lecture : synthèse documentaire
        lignes = [l.strip() for l in corps.splitlines() if l.strip()][:9]
        if not lignes:
            return "L'outil n'a renvoyé aucun résultat exploitable."
        entete = f"Voici ce que j'ai trouvé à propos de « {self._extraire_sujet_recherche(demande)} » :"
        return entete + "\n\n" + "\n".join(
            l if l.startswith(("-", "#", "[", "(")) else f"- {l}" for l in lignes
        ) + "\n\n(Synthèse produite par le modèle simulé de l'atelier.)"

    def _reponse_directe(self, demande: str, memoire: str) -> str:
        d = sans_accents(demande)
        faits = [l.strip("- ").strip() for l in memoire.splitlines() if l.strip().startswith("-")]
        rappel = (" Je garde en mémoire : " + " ; ".join(faits[:3]) + ".") if faits else ""

        if any(k in d for k in ["bonjour", "salut", "bonsoir", "coucou"]):
            return ("Bonjour ! Je suis votre agent conversationnel supervisé." + rappel +
                    " Je peux chercher de l'information, lire des notes et préparer des e-mails "
                    "— ces derniers passeront toujours par votre validation.")
        if any(k in d for k in ["appelle", "mon nom", "je travaille", "qui suis-je",
                                "de moi", "sur moi", "rappeler", "souviens", "memorise"]):
            if faits:
                return "Oui, voici ce que je sais de vous :\n" + "\n".join(f"- {f}" for f in faits)
            return ("Je n'ai encore rien mémorisé à votre sujet. Présentez-vous "
                    "(« je m'appelle… », « je travaille chez… ») et je le retiendrai.")
        if any(k in d for k in ["merci", "parfait", "super", "tres bien"]):
            return "Avec plaisir. Autre chose ?"
        if "?" in demande:
            return ("Je n'ai pas d'élément certain sur ce point sans consulter une source." + rappel +
                    " Demandez-moi une recherche (« cherche… ») et je consulterai la documentation.")
        return ("C'est noté." + rappel + " Dites-moi si vous voulez que je lance une recherche "
                "ou que je prépare un message.")


print("ModeleSimule défini.")

ModeleSimule défini.


In [ ]:
def construire_modele(verbeux: bool = True):
    """Retourne (modele, étiquette). Utilise un vrai fournisseur si une clé est présente."""
    if os.environ.get("OPENAI_API_KEY"):
        try:
            from langchain_openai import ChatOpenAI
            m = ChatOpenAI(model=os.environ.get("MODELE_OPENAI", "gpt-4.1-mini"), temperature=0)
            if verbeux:
                print(f"Modèle : OpenAI ({m.model_name}) — appels réseau réels.")
            return m, "openai"
        except Exception as e:
            print(f"[avertissement] OpenAI indisponible ({e}). Bascule sur le modèle simulé.")
    if os.environ.get("ANTHROPIC_API_KEY"):
        try:
            from langchain_anthropic import ChatAnthropic
            m = ChatAnthropic(model=os.environ.get("MODELE_ANTHROPIC", "claude-sonnet-4-5-20250929"),
                              temperature=0, max_tokens=1024)
            if verbeux:
                print("Modèle : Anthropic — appels réseau réels.")
            return m, "anthropic"
        except Exception as e:
            print(f"[avertissement] Anthropic indisponible ({e}). Bascule sur le modèle simulé.")
    if verbeux:
        print("Modèle : SIMULÉ (aucune clé API détectée).")
        print("   → Le notebook fonctionne intégralement hors-ligne.")
        print("   → Renseignez une clé en partie 2 pour utiliser un vrai LLM.")
    return ModeleSimule(), "simule"


modele, type_de_modele = construire_modele()

Modèle : SIMULÉ (aucune clé API détectée).
   → Le notebook fonctionne intégralement hors-ligne.
   → Renseignez une clé en partie 2 pour utiliser un vrai LLM.


In [ ]:
# Vérification : le modèle répond-il ? (sans outils liés, il ne peut que rédiger)
essai = modele.invoke([HumanMessage(content="Bonjour, peux-tu te présenter ?")])
print(texte_de(essai))

Bonjour ! Je suis votre agent conversationnel supervisé. Je peux chercher de l'information, lire des notes et préparer des e-mails — ces derniers passeront toujours par votre validation.


---
## Partie 4 — Les outils, classés par niveau de risque

C'est la décision d'architecture la plus importante de l'atelier :

| Outil | Nature | Effet | Validation humaine |
|---|---|---|---|
| `recherche_web` | lecture | aucun effet extérieur | **non** |
| `lire_note` | lecture | aucun effet extérieur | **non** |
| `enregistrer_preference` | écriture interne | mémoire de l'agent | **non** |
| `envoyer_email` | **effet de bord** | message parti, irréversible | **OUI** |
| `supprimer_document` | **effet de bord** | fichier perdu, irréversible | **OUI** |

> **Anti-patron à éviter** : faire valider *tous* les appels d'outils. L'opérateur, sollicité
> vingt fois par minute pour des recherches inoffensives, finit par cliquer « OK » sans lire :
> c'est le *rubber-stamping*. Une supervision humaine n'est efficace que si elle est **rare et
> motivée**. On ne fait valider que ce qui est **irréversible ou coûteux**.

### La cascade de recherche

`recherche_web` essaie successivement :
1. **Tavily** (si `TAVILY_API_KEY` est définie) — API de recherche pensée pour les agents ;
2. **DuckDuckGo** via `ddgs` — sans clé, mais tributaire du réseau et de quotas ;
3. **le corpus local** généré en partie 1 — toujours disponible.

Toute exception est rattrapée : l'outil ne peut pas faire échouer le graphe.

In [ ]:
import json
import shutil
from datetime import datetime, timezone

from langchain_core.tools import tool

# --- état partagé (simplification pédagogique assumée : voir l'exercice 3) ---
MAGASIN = None            # sera renseigné en partie 5 avec le Store LangGraph
ID_UTILISATEUR = "stephane"
AUTORISER_RESEAU = True   # passez à False pour forcer une recherche 100 % hors-ligne

# LA liste de référence : c'est elle qui déclenche la validation humaine.
OUTILS_SENSIBLES = {"envoyer_email", "supprimer_document"}


def _horodatage() -> str:
    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")


def _journaliser(fichier: str, enregistrement: dict) -> None:
    """Écrit une ligne JSON dans un fichier de sortie (boîte d'envoi ou piste d'audit)."""
    with (DOSSIERS["sortie"] / fichier).open("a", encoding="utf-8") as f:
        f.write(json.dumps(enregistrement, ensure_ascii=False) + "\n")


# ------------------------------------------------- sources de recherche
def _recherche_corpus_local(requete: str, k: int = 3) -> str:
    """Moteur de recherche minimal sur le corpus local : score par occurrences de mots."""
    mots = {m for m in re.findall(r"\w+", sans_accents(requete)) if len(m) > 3}
    scores = []
    for fichier in sorted(DOSSIERS["corpus"].glob("*.md")):
        texte = fichier.read_text(encoding="utf-8")
        score = sum(sans_accents(texte).count(m) for m in mots)
        scores.append((score, fichier, texte))
    scores.sort(key=lambda x: -x[0])
    retenus = [s for s in scores if s[0] > 0][:k] or scores[:1]
    blocs = []
    for score, fichier, texte in retenus:
        lignes = [l for l in texte.splitlines() if l.strip()]
        titre = lignes[0].lstrip("# ").strip()
        extrait = " ".join(lignes[1:7])[:420]
        blocs.append(f"[corpus local] {titre}\n{extrait}\n(source : {fichier.name}, pertinence {score})")
    return "\n\n".join(blocs)


def _recherche_tavily(requete: str, k: int = 3):
    cle = os.environ.get("TAVILY_API_KEY")
    if not cle:
        return None
    try:
        import requests
        reponse = requests.post("https://api.tavily.com/search",
                                json={"api_key": cle, "query": requete,
                                      "max_results": k, "search_depth": "basic"},
                                timeout=20)
        reponse.raise_for_status()
        resultats = reponse.json().get("results", [])
        if not resultats:
            return None
        return "\n\n".join(
            f"[web/tavily] {x.get('title', '(sans titre)')}\n{(x.get('content') or '')[:400]}\n"
            f"(source : {x.get('url', '')})" for x in resultats)
    except Exception:
        return None


def _recherche_duckduckgo(requete: str, k: int = 3):
    if not AUTORISER_RESEAU:
        return None
    try:
        from ddgs import DDGS
        with DDGS(timeout=15) as moteur:
            resultats = list(moteur.text(requete, region="fr-fr", max_results=k))
        if not resultats:
            return None
        return "\n\n".join(
            f"[web/duckduckgo] {x.get('title', '(sans titre)')}\n{(x.get('body') or '')[:400]}\n"
            f"(source : {x.get('href') or x.get('url', '')})" for x in resultats)
    except Exception:
        return None


# =========================== LES OUTILS DE L'AGENT ===========================
@tool
def recherche_web(requete: str) -> str:
    """Recherche des informations documentaires sur un sujet donné.

    Outil en LECTURE SEULE : aucune validation humaine n'est requise.
    """
    for source in (_recherche_tavily, _recherche_duckduckgo):
        resultat = source(requete)
        if resultat:
            return resultat
    return _recherche_corpus_local(requete)


@tool
def lire_note(sujet: str) -> str:
    """Lit une note interne de l'équipe à partir d'un sujet ou d'un nom de fichier.

    Outil en LECTURE SEULE : aucune validation humaine n'est requise.
    """
    mots = {m for m in re.findall(r"\w+", sans_accents(sujet)) if len(m) > 3}
    meilleur, score_max = None, -1
    for fichier in sorted(DOSSIERS["notes"].glob("*.md")):
        cible = sans_accents(fichier.name + " " + fichier.read_text(encoding="utf-8"))
        score = sum(cible.count(m) for m in mots)
        if score > score_max:
            meilleur, score_max = fichier, score
    if meilleur is None:
        return "Aucune note disponible."
    return f"[note interne : {meilleur.name}]\n{meilleur.read_text(encoding='utf-8')}"


@tool
def enregistrer_preference(cle: str, valeur: str) -> str:
    """Enregistre durablement une information sur l'utilisateur (mémoire longue).

    Outil en ÉCRITURE MAÎTRISÉE : il n'écrit que dans la mémoire de l'agent.
    """
    if MAGASIN is None:
        return "Mémoire longue indisponible."
    MAGASIN.put(("memoire", ID_UTILISATEUR), cle, {"valeur": valeur, "date": _horodatage()})
    return f"Mémorisé : {cle} = {valeur}"


@tool
def envoyer_email(destinataire: str, sujet: str, corps: str) -> str:
    """Envoie un e-mail. ACTION IRRÉVERSIBLE : validation humaine obligatoire.

    Dans cet atelier, l'envoi est simulé par une écriture dans sortie/boite_envoi.jsonl.
    """
    _journaliser("boite_envoi.jsonl", {"date": _horodatage(), "destinataire": destinataire,
                                       "sujet": sujet, "corps": corps})
    return (f"E-mail envoyé à {destinataire} — sujet « {sujet} » "
            f"({len(corps)} caractères). Copie dans sortie/boite_envoi.jsonl.")


@tool
def supprimer_document(nom_fichier: str) -> str:
    """Supprime un document du dossier documents/. ACTION IRRÉVERSIBLE :
    validation humaine obligatoire. Le fichier est déplacé dans corbeille/."""
    source = DOSSIERS["documents"] / Path(nom_fichier).name
    if not source.exists():
        return f"Fichier introuvable : {source.name}"
    shutil.move(str(source), str(DOSSIERS["corbeille"] / source.name))
    return f"Document supprimé : {source.name} (déplacé dans corbeille/)."


OUTILS = [recherche_web, lire_note, enregistrer_preference, envoyer_email, supprimer_document]

print(f"{len(OUTILS)} outils définis :")
for o in OUTILS:
    marque = "VALIDATION HUMAINE" if o.name in OUTILS_SENSIBLES else "exécution directe"
    print(f"  - {o.name:<24} [{marque}]")

5 outils définis :
  - recherche_web            [exécution directe]
  - lire_note                [exécution directe]
  - enregistrer_preference   [exécution directe]
  - envoyer_email            [VALIDATION HUMAINE]
  - supprimer_document       [VALIDATION HUMAINE]


In [ ]:
# Essai direct des outils, hors du graphe (on les appelle avec .invoke({...}))
print("=== recherche_web ===")
print(recherche_web.invoke({"requete": "human in the loop validation"})[:400], "...\n")

print("=== lire_note ===")
print(lire_note.invoke({"sujet": "checklist mise en production"})[:400], "...\n")

print("=== schéma transmis au LLM pour envoyer_email ===")
print(json.dumps(envoyer_email.args_schema.model_json_schema()["properties"],
                 ensure_ascii=False, indent=2))

=== recherche_web ===
[web/duckduckgo] Implement human-in-the-loop confirmation with Amazon Bedrock ...
Apr 9, 2025 · In this post, we explored two primary frameworks for implementing human validation in Amazon Bedrock Agents: user confirmation and return of control. Although these mechanisms serve similar oversight purposes, they address different validation needs and operate at distinct levels of the agent’s workflow ...

=== lire_note ===
[note interne : checklist_mise_en_production.md]
# Checklist mise en production d'un agent
1. Tous les outils à effet de bord passent par une validation humaine.
2. Le checkpointer est persistant (pas InMemorySaver).
3. Les identifiants sont lus depuis des variables d'environnement.
4. Chaque validation est journalisée (qui, quoi, quand).
5. Un délai d'expiration est prévu si l'humain ne répond pa ...

=== schéma transmis au LLM pour envoyer_email ===
{
  "destinataire": {
    "title": "Destinataire",
    "type": "string"
  },
  "sujet": {
    "tit

---
## Partie 5 — Les deux mémoires

LangGraph distingue nettement deux persistances, et les confondre est une erreur classique.

| | Mémoire **courte** | Mémoire **longue** |
|---|---|---|
| Objet LangGraph | `Checkpointer` (`SqliteSaver`) | `Store` (`SqliteStore`) |
| Portée | **une** conversation (`thread_id`) | **un** utilisateur, toutes conversations |
| Contient | l'historique des messages, l'état du graphe | des faits durables (nom, employeur, préférences) |
| Sans elle | pas de suite de conversation… **et pas de `interrupt()`** | l'agent redécouvre l'utilisateur à chaque fois |

> **Point crucial** : `interrupt()` **exige** un checkpointer. La pause consiste à figer l'état
> complet du graphe sur disque, puis à ressortir de `stream()`. Sans checkpointer, il n'y a rien
> à figer, donc rien à reprendre.

Les deux bases sont des fichiers SQLite : fermez le notebook, rouvrez-le, **tout est là**.
Attention à un détail d'implémentation : le `Store` gère lui-même ses transactions, sa connexion
doit donc être ouverte avec `isolation_level=None` (mode *autocommit*).

In [ ]:
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore


def ouvrir_memoires():
    """Ouvre la mémoire courte (checkpointer) et la mémoire longue (store), toutes deux sur disque."""
    # --- mémoire courte : l'historique et l'état du graphe, par thread_id
    connexion_courte = sqlite3.connect(str(DOSSIERS["memoire"] / "memoire_courte.sqlite"),
                                       check_same_thread=False)
    checkpointer = SqliteSaver(connexion_courte)
    checkpointer.setup()

    # --- mémoire longue : les faits durables, par utilisateur
    #     isolation_level=None : le Store pilote lui-même ses transactions.
    connexion_longue = sqlite3.connect(str(DOSSIERS["memoire"] / "memoire_longue.sqlite"),
                                       check_same_thread=False, isolation_level=None)
    magasin = SqliteStore(connexion_longue)
    magasin.setup()
    return checkpointer, magasin


checkpointer, magasin = ouvrir_memoires()

# on branche la mémoire longue sur les outils définis en partie 4
MAGASIN = magasin
ID_UTILISATEUR = "stephane"

print("Mémoire courte :", DOSSIERS["memoire"] / "memoire_courte.sqlite")
print("Mémoire longue :", DOSSIERS["memoire"] / "memoire_longue.sqlite")

Mémoire courte : /content/atelier_hitl/memoire/memoire_courte.sqlite
Mémoire longue : /content/atelier_hitl/memoire/memoire_longue.sqlite


In [ ]:
# ------------- lecture / écriture de la mémoire longue -------------
def lire_memoire_longue(magasin, id_utilisateur: str) -> str:
    """Renvoie les faits mémorisés, un par ligne, prêts à être injectés dans l'invite système."""
    if magasin is None:
        return ""
    elements = magasin.search(("memoire", id_utilisateur), limit=20)
    return "\n".join(
        f"- {e.value.get('valeur') if isinstance(e.value, dict) else e.value}" for e in elements)


# (clé, motif, gabarit) — la clé rend la mémoire idempotente : une nouvelle
# déclaration met à jour le fait existant au lieu d'en empiler un doublon.
# Les motifs s'appliquent au texte passé par sans_accents() : ils sont donc
# écrits en minuscules et SANS accents.
MOTIFS_MEMOIRE = [
    ("nom", r"je m['’ ]appelle ([a-z\-]{2,20}(?: [a-z\-]{2,20})?)",
     "L'utilisateur s'appelle {}"),
    ("employeur", r"je travaille (?:chez|pour|au sein de) ([a-z0-9\-&\.]{2,30}(?: [a-z0-9\-&\.]{2,30})?)",
     "L'utilisateur travaille chez {}"),
    ("metier", r"je suis (formateur|formatrice|developpeur|developpeuse|ingenieur|ingenieure|"
               r"consultant|consultante|enseignant|enseignante|architecte|data engineer)",
     "L'utilisateur est {}"),
    ("preference", r"je prefere ([a-z0-9\-' ]{3,60})", "L'utilisateur préfère {}"),
    ("projet", r"mon projet s['’ ]appelle ([a-z0-9\-]{2,30})",
     "Le projet de l'utilisateur s'appelle {}"),
    ("email", r"mon (?:adresse )?(?:e-?mail|courriel) est ([\w\.\-\+]+@[\w\.\-]+\.\w+)",
     "L'adresse e-mail de l'utilisateur est {}"),
]

# On n'extrait rien d'une question : « comment je m'appelle ? » n'est pas une déclaration.
AMORCES_QUESTION = ("comment", "qui", "quel", "quelle", "est-ce que",
                    "peux-tu", "peux tu", "sais-tu", "sais tu")


def extraire_faits(texte: str) -> list:
    """Retourne une liste de couples (clé, fait) extraits d'une déclaration de l'utilisateur."""
    t = sans_accents(texte)                 # même longueur que `texte` : les positions coïncident
    if t.strip().endswith("?") or t.strip().startswith(AMORCES_QUESTION):
        return []
    faits = []
    for cle, motif, gabarit in MOTIFS_MEMOIRE:
        trouve = re.search(motif, t)
        if not trouve:
            continue
        # on relit la valeur dans le texte d'origine pour conserver accents et majuscules
        valeur = texte[trouve.start(1):trouve.end(1)].strip(" .,;'")
        # on coupe sur une conjonction : « stephane et je travaille… » → « stephane »
        valeur = re.split(r"\s+(?:et|mais|donc|puis|ou)\s+", valeur)[0].strip()
        valeur = re.sub(r"\s+(?:et|mais|donc|puis|ou|le|la|les|de|du)$", "", valeur).strip()
        if valeur:
            if cle in ("nom", "employeur", "projet"):
                valeur = valeur.title()
            faits.append((cle, gabarit.format(valeur)))
    return faits


# démonstration hors graphe
for phrase in ["Je m'appelle Stéphane et je travaille chez Formatis.",
               "Je suis formateur, je préfère les explications avec des schémas.",
               "Comment je m'appelle ?"]:
    print(f"{phrase!r}\n   → {extraire_faits(phrase) or 'aucun fait (ce n’est pas une déclaration)'}")

"Je m'appelle Stéphane et je travaille chez Formatis."
   → [('nom', "L'utilisateur s'appelle Stéphane"), ('employeur', "L'utilisateur travaille chez Formatis")]
'Je suis formateur, je préfère les explications avec des schémas.'
   → [('metier', "L'utilisateur est formateur"), ('preference', "L'utilisateur préfère les explications avec des schémas")]
"Comment je m'appelle ?"
   → aucun fait (ce n’est pas une déclaration)


---
## Partie 6 — L'état, les nœuds, et le point d'arrêt humain

### 6.0 Rappel : ce que le HITL corrige dans ReACT

La boucle **ReACT** vue en cours — *raisonner, agir, observer, recommencer* — est exactement le
cycle `agent → outils → agent` de notre graphe. Sa limite structurelle est dans sa définition
même : **rien, entre « décider d'agir » et « agir », n'est prévu pour un tiers**. L'agent observe
le résultat de son action *après* qu'elle a eu lieu. Pour une recherche documentaire, c'est sans
conséquence ; pour un virement ou une suppression, c'est irrattrapable.

Le Human-in-the-Loop ne remplace pas ReACT : il **ouvre sa boucle** à un endroit choisi, entre la
décision et l'exécution.

Autre limite à garder en tête, traitée en cours sous l'angle du **contrôle de la récursivité** :
une boucle ReACT peut ne jamais s'arrêter (l'agent rappelle indéfiniment un outil qui échoue).
LangGraph plafonne cela par `recursion_limit` — nous le vérifierons en fin de partie 7.

### 6.1 L'état

L'état est un `TypedDict` partagé par tous les nœuds. Chaque champ peut recevoir un **réducteur**
qui dit comment fusionner l'ancienne et la nouvelle valeur :

- `messages` est annoté par `add_messages` : les nouveaux messages sont **ajoutés** à la liste,
  et — détail capital pour le patron « modifier » — **un message renvoyé avec un `id` déjà
  présent remplace l'ancien**.
- `journal` utilise `operator.add` : les décisions humaines s'accumulent.
- `contexte_memoire` est une simple chaîne, écrasée à chaque tour.

### 6.2 Le graphe

```
        START
          │
   charger_memoire        ← lit la mémoire longue
          │
        agent  ◀──────────────┐
          │                   │
    ┌─────┴──────┬────────┐   │
    │            │        │   │
 revision_    outils   memoriser
  humaine        │        │   │
    │ ┌──────────┘        │   │
    │ │                   │   │
    └─┴───────────────────┘   │
      (approuvé/modifié)──────┘
          │
         END
```

### 6.3 Le cœur : `interrupt()`

```python
decision = interrupt({"outil": "envoyer_email", "arguments": {...}})
```

Trois choses se produisent, dans cet ordre :

1. L'exécution du nœud **s'arrête net** ; LangGraph lève une exception interne `GraphInterrupt`.
2. L'état complet est **écrit dans le checkpointer**. Le processus peut mourir : rien n'est perdu.
3. `stream()` rend la main à l'appelant, avec la charge utile passée à `interrupt()`.

Plus tard — une seconde ou trois jours après — on relance le graphe avec
`Command(resume=<décision>)`. **Le nœud est ré-exécuté depuis son début**, mais cette fois
`interrupt()` retourne immédiatement la valeur fournie.

> **Conséquence pratique, souvent source de bugs** : tout ce qui précède l'appel `interrupt()`
> dans le nœud sera exécuté **deux fois**. On ne place donc jamais d'effet de bord (envoi, écriture)
> avant un `interrupt()` — seulement de la préparation idempotente.

### 6.4 Les quatre décisions implémentées

| Décision | Ce que fait le nœud | Suite du graphe |
|---|---|---|
| `approuver` | rien : l'appel part tel quel | → `outils` |
| `modifier` | remplace le message de l'agent par une version aux arguments corrigés (même `id`) | → `outils` |
| `refuser` | fabrique un `ToolMessage` « action refusée » | → `agent` |
| `repondre` | fabrique un `ToolMessage` contenant la consigne de l'opérateur | → `agent` |

Dans les deux derniers cas, **l'outil n'est jamais exécuté** — et pourtant le modèle reçoit bien
une réponse à son `tool_call`, ce qu'exige le protocole des appels de fonctions.

In [ ]:
import operator
from typing import Annotated, Literal

from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt


class EtatAgent(TypedDict):
    """L'état partagé par tous les nœuds du graphe."""
    messages: Annotated[list, add_messages]        # historique (ajout + remplacement par id)
    contexte_memoire: str                          # faits issus de la mémoire longue
    journal: Annotated[list, operator.add]         # décisions humaines de la session


INVITE_SYSTEME = """Tu es un assistant professionnel francophone qui travaille sous supervision humaine.
Règles :
- Tu réponds en français, de manière concise et factuelle.
- Pour toute information que tu ne connais pas avec certitude, tu utilises l'outil recherche_web.
- Les outils envoyer_email et supprimer_document ont des effets irréversibles : tu les proposes,
  c'est l'humain qui autorise. Tu ne promets jamais qu'une action est faite avant sa validation.
- Tu tiens compte des faits mémorisés ci-dessous s'ils sont pertinents."""

print("État et invite système définis.")

État et invite système définis.


In [ ]:
def construire_graphe(modele, checkpointer, magasin, id_utilisateur="stephane", tracer=True):
    """Assemble et compile le graphe de l'agent Human-in-the-Loop."""

    modele_outille = modele.bind_tools(OUTILS)
    noeud_outils = ToolNode(OUTILS)

    # ------------------------------------------------------------- nœud 1
    def charger_memoire(etat: EtatAgent) -> dict:
        """Lit la mémoire longue et la dépose dans l'état (elle n'est pas dans l'historique)."""
        return {"contexte_memoire": lire_memoire_longue(magasin, id_utilisateur)}

    # ------------------------------------------------------------- nœud 2
    def agent(etat: EtatAgent) -> dict:
        """Appelle le LLM. L'invite système est reconstruite à chaque tour : elle n'encombre
        jamais l'historique persisté."""
        invite = INVITE_SYSTEME
        if etat.get("contexte_memoire"):
            invite += "\n\nFaits mémorisés sur l'utilisateur :\n" + etat["contexte_memoire"]
        reponse = modele_outille.invoke([SystemMessage(content=invite)] + etat["messages"])
        return {"messages": [reponse]}

    # --------------------------------------------------------- aiguillage
    def appels_sensibles(message) -> list:
        return [a for a in (message.tool_calls or []) if a["name"] in OUTILS_SENSIBLES]

    def aiguiller(etat: EtatAgent) -> Literal["revision_humaine", "outils", "memoriser"]:
        """LA règle d'architecture : un appel sensible passe par l'humain, les autres non."""
        dernier = etat["messages"][-1]
        if not isinstance(dernier, AIMessage) or not dernier.tool_calls:
            return "memoriser"                       # pas d'outil demandé → fin du tour
        if appels_sensibles(dernier):
            return "revision_humaine"                # effet de bord → PAUSE
        return "outils"                              # lecture seule → exécution directe

    # ============== nœud 3 : LE POINT D'ARRÊT HUMAIN (cœur de l'atelier) ==============
    def revision_humaine(etat: EtatAgent) -> Command[Literal["outils", "agent"]]:
        dernier = etat["messages"][-1]
        en_attente = list(dernier.tool_calls)
        sensibles = appels_sensibles(dernier)

        # >>> L'EXÉCUTION S'ARRÊTE ICI. L'état est écrit sur disque. <<<
        # Au retour (Command(resume=...)), `decision` contient la réponse de l'humain.
        decision = interrupt({
            "type": "validation_humaine",
            "message_agent": texte_de(dernier),
            "appels_en_attente": [{"outil": a["name"], "arguments": a["args"]} for a in en_attente],
            "outil_sensible": sensibles[0]["name"],
            "arguments": sensibles[0]["args"],
            "actions_possibles": {
                "approuver": "exécuter tel quel",
                "modifier": "corriger les arguments puis exécuter → fournir 'arguments'",
                "refuser": "ne pas exécuter → fournir 'motif'",
                "repondre": "ne pas exécuter et renvoyer une consigne → fournir 'message'",
            },
        })

        if isinstance(decision, str):                # tolère reprendre(graphe, "approuver", ...)
            decision = {"action": decision}
        action = (decision or {}).get("action", "refuser")
        trace = {"outil": sensibles[0]["name"], "action": action,
                 "arguments_initiaux": sensibles[0]["args"],
                 "validateur": decision.get("validateur", id_utilisateur),
                 "date": _horodatage()}

        # ---- 1) APPROUVER : on laisse partir l'appel tel quel
        if action == "approuver":
            _journaliser("journal_validations.jsonl", {**trace, "resultat": "execute"})
            if tracer:
                print(f"   [journal] {trace['validateur']} a APPROUVÉ {trace['outil']}")
            return Command(goto="outils", update={"journal": [trace]})

        # ---- 2) MODIFIER : on réécrit le message de l'agent (même id ⇒ remplacement)
        if action == "modifier":
            nouveaux_args = {**sensibles[0]["args"], **decision.get("arguments", {})}
            appels_corriges = [{**a, "args": nouveaux_args} if a["id"] == sensibles[0]["id"] else a
                               for a in en_attente]
            message_corrige = AIMessage(id=dernier.id, content=texte_de(dernier),
                                        tool_calls=appels_corriges)
            trace["arguments_finaux"] = nouveaux_args
            _journaliser("journal_validations.jsonl",
                         {**trace, "resultat": "execute_apres_correction"})
            if tracer:
                print(f"   [journal] {trace['validateur']} a MODIFIÉ puis approuvé {trace['outil']}")
            return Command(goto="outils",
                           update={"messages": [message_corrige], "journal": [trace]})

        # ---- 3) RÉPONDRE : l'outil n'est pas exécuté, une consigne remonte au modèle
        if action == "repondre":
            consigne = decision.get("message", "Reprends la demande différemment.")
            retours = [ToolMessage(content=f"Action non exécutée. Consigne de l'opérateur : {consigne}",
                                   tool_call_id=a["id"], name=a["name"]) for a in en_attente]
            _journaliser("journal_validations.jsonl",
                         {**trace, "resultat": "consigne", "message": consigne})
            if tracer:
                print(f"   [journal] {trace['validateur']} a renvoyé une CONSIGNE sur {trace['outil']}")
            return Command(goto="agent", update={"messages": retours, "journal": [trace]})

        # ---- 4) REFUSER (cas par défaut, le plus sûr)
        motif = decision.get("motif", "refus de l'opérateur, sans motif précisé")
        retours = [ToolMessage(
            content=f"Action REFUSÉE par l'opérateur humain. Motif : {motif}. "
                    f"N'insiste pas et explique la situation à l'utilisateur.",
            tool_call_id=a["id"], name=a["name"]) for a in en_attente]
        _journaliser("journal_validations.jsonl", {**trace, "resultat": "refuse", "motif": motif})
        if tracer:
            print(f"   [journal] {trace['validateur']} a REFUSÉ {trace['outil']}")
        return Command(goto="agent", update={"messages": retours, "journal": [trace]})

    # ------------------------------------------------------------- nœud 5
    def memoriser(etat: EtatAgent) -> dict:
        """Fin de tour : extrait d'éventuels faits durables et les écrit en mémoire longue."""
        derniere_demande = ""
        for m in reversed(etat["messages"]):
            if isinstance(m, HumanMessage):
                derniere_demande = texte_de(m)
                break
        deja_connu = etat.get("contexte_memoire", "")
        for cle, fait in extraire_faits(derniere_demande):
            if fait not in deja_connu and magasin is not None:
                magasin.put(("memoire", id_utilisateur), cle, {"valeur": fait})
                if tracer:
                    print(f"   [mémoire longue] {cle} = {fait}")
        return {}

    # --------------------------------------------------- assemblage du graphe
    g = StateGraph(EtatAgent)
    g.add_node("charger_memoire", charger_memoire)
    g.add_node("agent", agent)
    g.add_node("revision_humaine", revision_humaine)
    g.add_node("outils", noeud_outils)
    g.add_node("memoriser", memoriser)

    g.add_edge(START, "charger_memoire")
    g.add_edge("charger_memoire", "agent")
    g.add_conditional_edges("agent", aiguiller, {
        "revision_humaine": "revision_humaine",
        "outils": "outils",
        "memoriser": "memoriser",
    })
    g.add_edge("outils", "agent")
    g.add_edge("memoriser", END)

    # compile() reçoit le checkpointer : SANS LUI, interrupt() est impossible.
    return g.compile(checkpointer=checkpointer, store=magasin)


graphe = construire_graphe(modele, checkpointer, magasin, ID_UTILISATEUR)
print("Graphe compilé.")

Graphe compilé.


---
## Partie 7 — Lecture du graphe et fonctions d'exécution

Affichons d'abord la topologie réellement compilée, puis écrivons deux fonctions qui serviront
dans tous les scénarios :

- `discuter(graphe, texte, thread)` : envoie un message. Retourne `None` si le tour s'est terminé,
  ou **la charge utile de l'interruption** si l'agent attend une validation.
- `reprendre(graphe, decision, thread)` : relance l'exécution avec la décision de l'humain.

In [ ]:
# Représentation ASCII (fonctionne hors-ligne, contrairement au rendu PNG via mermaid.ink)
print(graphe.get_graph().draw_ascii())

                                    +-----------+                                
                                    | __start__ |                                
                                    +-----------+                                
                                           *                                     
                                           *                                     
                                           *                                     
                                  +-----------------+                            
                                  | charger_memoire |                            
                                  +-----------------+                            
                                           *                                     
                                           *                                     
                                           *                                     
                

### Contrôle de la récursivité

Le graphe contient un cycle (`agent → outils → agent`). Sans garde-fou, un agent qui s'entête sur
un outil en échec bouclerait indéfiniment. LangGraph plafonne le nombre de **super-étapes** par
exécution : `recursion_limit`, **25 par défaut**, réglable dans la configuration.

Vérifions-le en l'abaissant volontairement à 2 — soit moins que le minimum nécessaire
(`charger_memoire` → `agent` → `outils` → `agent` → `memoriser`).

In [ ]:
from langgraph.errors import GraphRecursionError

config_bridee = {"configurable": {"thread_id": "demo-recursion"}, "recursion_limit": 2}
try:
    graphe.invoke({"messages": [HumanMessage(content="Cherche ce qu'est le Human-in-the-Loop")]},
                  config_bridee)
    print("Terminé sans atteindre la limite.")
except GraphRecursionError as e:
    print("GraphRecursionError levée, comme attendu :")
    print(f"   {str(e).splitlines()[0]}")
    print("\n→ En production, on règle recursion_limit selon la profondeur maximale légitime du "
          "graphe,\n   et on traite cette exception comme un incident à remonter, pas comme un bug.")

GraphRecursionError levée, comme attendu :
   Recursion limit of 2 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.

→ En production, on règle recursion_limit selon la profondeur maximale légitime du graphe,
   et on traite cette exception comme un incident à remonter, pas comme un bug.


In [ ]:
import textwrap

LARGEUR = 90


def afficher_pause(charge: dict) -> None:
    """Met en forme la demande de validation destinée à l'opérateur humain."""
    print()
    print("=" * LARGEUR)
    print("PAUSE — VALIDATION HUMAINE REQUISE")
    print("=" * LARGEUR)
    print(f"  Outil demandé : {charge['outil_sensible']}")
    print("  Arguments proposés par l'agent :")
    for cle, valeur in charge["arguments"].items():
        print(f"      - {cle:<14}: " + str(valeur).replace("\n", "\n" + " " * 24))
    if charge.get("message_agent"):
        print(f"  Message de l'agent : {charge['message_agent']}")
    print("  Décisions possibles :")
    for nom, description in charge["actions_possibles"].items():
        print(f"      - {nom:<10} : {description}")
    print("=" * LARGEUR)


def _afficher_nouveaux(messages, deja_vus: set) -> None:
    for m in messages:
        cle = id(m) if m.id is None else m.id
        if cle in deja_vus:
            continue
        deja_vus.add(cle)
        if isinstance(m, HumanMessage):
            print(f"\n[UTILISATEUR] {texte_de(m)}")
        elif isinstance(m, ToolMessage):
            extrait = texte_de(m).strip().replace("\n", " ")[:150]
            print(f"   [résultat outil : {m.name}] {extrait}…")
        elif isinstance(m, AIMessage):
            for a in (m.tool_calls or []):
                print(f"   [appel outil] {a['name']}({json.dumps(a['args'], ensure_ascii=False)[:140]})")
            if texte_de(m).strip():
                print(f"[AGENT] {texte_de(m).strip()}")


def _executer(graphe, entree, config, afficher: bool):
    deja_vus = set()
    etat_initial = graphe.get_state(config)
    if etat_initial and etat_initial.values.get("messages"):
        for m in etat_initial.values["messages"]:
            deja_vus.add(id(m) if m.id is None else m.id)

    for evenement in graphe.stream(entree, config, stream_mode="values"):
        if afficher and isinstance(evenement, dict) and evenement.get("messages"):
            _afficher_nouveaux(evenement["messages"], deja_vus)

    etat = graphe.get_state(config)
    if etat.interrupts:                       # le graphe est en pause
        charge = etat.interrupts[0].value
        if afficher:
            afficher_pause(charge)
        return charge
    return None


def discuter(graphe, texte: str, thread: str, afficher: bool = True):
    """Envoie un message à l'agent. Retourne la charge utile de l'interruption, ou None."""
    config = {"configurable": {"thread_id": thread}}
    return _executer(graphe, {"messages": [HumanMessage(content=texte)]}, config, afficher)


def reprendre(graphe, decision, thread: str, afficher: bool = True):
    """Reprend une exécution en pause avec la décision de l'humain."""
    config = {"configurable": {"thread_id": thread}}
    if afficher:
        print(f"\n>>> DÉCISION HUMAINE : {json.dumps(decision, ensure_ascii=False)}")
    return _executer(graphe, Command(resume=decision), config, afficher)


def resume_etat(graphe, thread: str) -> None:
    etat = graphe.get_state({"configurable": {"thread_id": thread}})
    print(f"thread         : {thread}")
    print(f"messages       : {len(etat.values.get('messages', []))}")
    print(f"prochain nœud  : {', '.join(etat.next) if etat.next else '(fin de tour)'}")
    print(f"en pause       : {'OUI' if etat.interrupts else 'non'}")
    print(f"validations    : {len(etat.values.get('journal', []))}")


print("Fonctions d'exécution prêtes.")

Fonctions d'exécution prêtes.


---
## Partie 8 — Les scénarios guidés

Six scénarios, du plus simple au plus complet. **Exécutez-les dans l'ordre** : ils partagent
la mémoire longue de l'utilisateur.

### Scénario 1 — Conversation et mémoire longue

L'utilisateur se présente. Le nœud `memoriser` extrait les faits durables et les écrit dans
`SqliteStore`. Aucun outil sensible : **aucune pause**.

In [ ]:
discuter(graphe, "Bonjour ! Je m'appelle Stéphane et je travaille chez Formatis.", "atelier-1")


[UTILISATEUR] Bonjour ! Je m'appelle Stéphane et je travaille chez Formatis.
[AGENT] Bonjour ! Je suis votre agent conversationnel supervisé. Je peux chercher de l'information, lire des notes et préparer des e-mails — ces derniers passeront toujours par votre validation.
   [mémoire longue] nom = L'utilisateur s'appelle Stéphane
   [mémoire longue] employeur = L'utilisateur travaille chez Formatis


In [ ]:
# Nouveau message, même conversation : l'agent se souvient (mémoire courte + longue)
discuter(graphe, "Peux-tu me rappeler ce que tu sais de moi ?", "atelier-1")


[UTILISATEUR] Peux-tu me rappeler ce que tu sais de moi ?
[AGENT] Oui, voici ce que je sais de vous :
- L'utilisateur s'appelle Stéphane
- L'utilisateur travaille chez Formatis


### Scénario 2 — Un outil en lecture seule ne doit **pas** interrompre

`recherche_web` n'a aucun effet extérieur : l'aiguillage l'envoie directement vers `outils`.
C'est le contre-exemple qui donne son sens au reste : on ne fait valider que ce qui est
irréversible.

In [ ]:
resultat = discuter(graphe, "Cherche des informations sur le Human-in-the-Loop en LangGraph", "atelier-1")
print("\n>>> Interruption déclenchée :", "OUI" if resultat else "NON (attendu)")


[UTILISATEUR] Cherche des informations sur le Human-in-the-Loop en LangGraph
   [appel outil] recherche_web({"requete": "le Human-in-the-Loop en LangGraph"})
[AGENT] Je lance une recherche documentaire.
   [résultat outil : recherche_web] [web/duckduckgo] Human-in-the-Loop with LangGraph: A Beginner’s Guide | by Sangeethasaravanan | Medium May 7, 2025 - from typing import TypedDict, Ann…
[AGENT] Voici ce que j'ai trouvé à propos de « le Human-in-the-Loop en LangGraph » :

[web/duckduckgo] Human-in-the-Loop with LangGraph: A Beginner’s Guide | by Sangeethasaravanan | Medium
- May 7, 2025 - from typing import TypedDict, Annotated, Literal, Dict, Any from langgraph.graph import StateGraph, END import json # Define the state schema class HumanInTheLoopState(TypedDict): question: str # User's original question ai_draft: str # AI's initial response draft human_feedback: str # Human reviewer's feedback final_response: str # Final response after incorporating feedback # Creat
(source : https:

### Scénario 3 — `approuver`

Premier vrai point d'arrêt. L'agent prépare un e-mail ; `discuter()` **rend la main** avec la
charge utile de l'interruption au lieu d'exécuter l'outil.

Observez l'état du graphe entre les deux cellules : `prochain nœud` vaut `revision_humaine`,
et `en pause` vaut `OUI`. **L'e-mail n'est pas parti.**

In [ ]:
demande = discuter(graphe,
                   "Envoie un mail à leila@formatis.fr au sujet de la checklist de mise en production",
                   "atelier-2")


[UTILISATEUR] Envoie un mail à leila@formatis.fr au sujet de la checklist de mise en production
   [appel outil] envoyer_email({"destinataire": "leila@formatis.fr", "sujet": "Checklist de mise en production", "corps": "Bonjour,\n\nFaisant suite à votre demande concer)
[AGENT] J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.

PAUSE — VALIDATION HUMAINE REQUISE
  Outil demandé : envoyer_email
  Arguments proposés par l'agent :
      - destinataire  : leila@formatis.fr
      - sujet         : Checklist de mise en production
      - corps         : Bonjour,
                        
                        Faisant suite à votre demande concernant checklist de mise en production, voici le message préparé automatiquement par l'agent.
                        
                        Bien cordialement,
                        Votre agent.
  Message de l'agent : J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.
  Décisions possibles :
      -

In [ ]:
# État du graphe PENDANT la pause — tout est figé sur disque
resume_etat(graphe, "atelier-2")
print("\nBoîte d'envoi :", (DOSSIERS['sortie'] / 'boite_envoi.jsonl').read_text().strip() or "(vide — rien n'est parti)")

thread         : atelier-2
messages       : 2
prochain nœud  : revision_humaine
en pause       : OUI
validations    : 0

Boîte d'envoi : (vide — rien n'est parti)


In [ ]:
reprendre(graphe, {"action": "approuver", "validateur": "stephane"}, "atelier-2")


>>> DÉCISION HUMAINE : {"action": "approuver", "validateur": "stephane"}
   [journal] stephane a APPROUVÉ envoyer_email
   [résultat outil : envoyer_email] E-mail envoyé à leila@formatis.fr — sujet « Checklist de mise en production » (170 caractères). Copie dans sortie/boite_envoi.jsonl.…
[AGENT] C'est fait : E-mail envoyé à leila@formatis.fr — sujet « Checklist de mise en production » (170 caractères). Copie dans sortie/boite_envoi.jsonl.


In [ ]:
print("Boîte d'envoi après validation :")
for ligne in (DOSSIERS["sortie"] / "boite_envoi.jsonl").read_text().strip().splitlines():
    envoi = json.loads(ligne)
    print(f"  → {envoi['destinataire']} | {envoi['sujet']} | {envoi['date']}")

Boîte d'envoi après validation :
  → leila@formatis.fr | Checklist de mise en production | 2026-09-20T21:35:03+00:00


### Scénario 4 — `modifier`

Le patron le plus utile en production : le modèle propose, l'humain **corrige**, puis l'action part.
Ici l'opérateur rectifie l'adresse du destinataire et réécrit le corps du message.

Techniquement, le nœud renvoie un `AIMessage` portant **le même `id`** que l'original : le
réducteur `add_messages` le **remplace** dans l'historique. Le `ToolNode` exécute donc la version
corrigée, et l'historique reste cohérent.

In [ ]:
demande = discuter(graphe, "Envoie un mail à marc@formatis.fr au sujet de la réunion Atlas", "atelier-3")


[UTILISATEUR] Envoie un mail à marc@formatis.fr au sujet de la réunion Atlas
   [appel outil] envoyer_email({"destinataire": "marc@formatis.fr", "sujet": "Réunion Atlas", "corps": "Bonjour,\n\nFaisant suite à votre demande concernant réunion Atlas,)
[AGENT] J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.

PAUSE — VALIDATION HUMAINE REQUISE
  Outil demandé : envoyer_email
  Arguments proposés par l'agent :
      - destinataire  : marc@formatis.fr
      - sujet         : Réunion Atlas
      - corps         : Bonjour,
                        
                        Faisant suite à votre demande concernant réunion Atlas, voici le message préparé automatiquement par l'agent.
                        
                        Bien cordialement,
                        Votre agent.
  Message de l'agent : J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.
  Décisions possibles :
      - approuver  : exécuter tel quel
      - modifier   : cor

In [ ]:
reprendre(graphe, {
    "action": "modifier",
    "validateur": "stephane",
    "arguments": {
        "destinataire": "marc.durand@formatis.fr",
        "corps": "Bonjour Marc,\n\nPour rappel, la livraison du prototype HITL est fixée au "
                 "30 septembre. Peux-tu confirmer le choix du checkpointer ?\n\nStéphane",
    },
}, "atelier-3")


>>> DÉCISION HUMAINE : {"action": "modifier", "validateur": "stephane", "arguments": {"destinataire": "marc.durand@formatis.fr", "corps": "Bonjour Marc,\n\nPour rappel, la livraison du prototype HITL est fixée au 30 septembre. Peux-tu confirmer le choix du checkpointer ?\n\nStéphane"}}
   [journal] stephane a MODIFIÉ puis approuvé envoyer_email
   [résultat outil : envoyer_email] E-mail envoyé à marc.durand@formatis.fr — sujet « Réunion Atlas » (140 caractères). Copie dans sortie/boite_envoi.jsonl.…
[AGENT] C'est fait : E-mail envoyé à marc.durand@formatis.fr — sujet « Réunion Atlas » (140 caractères). Copie dans sortie/boite_envoi.jsonl.


In [ ]:
dernier = json.loads((DOSSIERS["sortie"] / "boite_envoi.jsonl").read_text().strip().splitlines()[-1])
print("Destinataire réellement utilisé :", dernier["destinataire"])
print("Corps réellement envoyé :\n" + textwrap.indent(dernier["corps"], "   "))

Destinataire réellement utilisé : marc.durand@formatis.fr
Corps réellement envoyé :
   Bonjour Marc,

   Pour rappel, la livraison du prototype HITL est fixée au 30 septembre. Peux-tu confirmer le choix du checkpointer ?

   Stéphane


### Scénario 5 — `refuser`

L'agent propose de supprimer un **contrat signé**. L'opérateur refuse, avec motif.

Deux points à observer :
1. Le fichier est toujours là — l'outil n'a **jamais** été exécuté.
2. Le modèle reçoit malgré tout un `ToolMessage` : le protocole des appels de fonctions impose
   qu'à chaque `tool_call` corresponde une réponse. Ce message porte le refus et sa raison, ce qui
   permet à l'agent de l'expliquer à l'utilisateur au lieu de réessayer en boucle.

In [ ]:
demande = discuter(graphe, "Supprime le fichier contrat_client_signe.txt", "atelier-4")


[UTILISATEUR] Supprime le fichier contrat_client_signe.txt
   [appel outil] supprimer_document({"nom_fichier": "contrat_client_signe.txt"})
[AGENT] Je propose de supprimer « contrat_client_signe.txt ». Cette action demande votre validation.

PAUSE — VALIDATION HUMAINE REQUISE
  Outil demandé : supprimer_document
  Arguments proposés par l'agent :
      - nom_fichier   : contrat_client_signe.txt
  Message de l'agent : Je propose de supprimer « contrat_client_signe.txt ». Cette action demande votre validation.
  Décisions possibles :
      - approuver  : exécuter tel quel
      - modifier   : corriger les arguments puis exécuter → fournir 'arguments'
      - refuser    : ne pas exécuter → fournir 'motif'
      - repondre   : ne pas exécuter et renvoyer une consigne → fournir 'message'


In [ ]:
reprendre(graphe, {"action": "refuser", "validateur": "stephane",
                   "motif": "document contractuel à conserver 10 ans"}, "atelier-4")


>>> DÉCISION HUMAINE : {"action": "refuser", "validateur": "stephane", "motif": "document contractuel à conserver 10 ans"}
   [journal] stephane a REFUSÉ supprimer_document
   [résultat outil : supprimer_document] Action REFUSÉE par l'opérateur humain. Motif : document contractuel à conserver 10 ans. N'insiste pas et explique la situation à l'utilisateur.…
[AGENT] L'action n'a pas été effectuée : elle a été refusée lors de la validation humaine (motif : document contractuel à conserver 10 ans). Je ne la relance pas. Dites-moi comment vous souhaitez procéder.


In [ ]:
existe = (DOSSIERS["documents"] / "contrat_client_signe.txt").exists()
print("contrat_client_signe.txt toujours présent :", "OUI (le refus a bien bloqué l'action)" if existe else "NON — PROBLÈME")
print("Contenu de corbeille/ :", [f.name for f in DOSSIERS["corbeille"].iterdir()] or "(vide)")

contrat_client_signe.txt toujours présent : OUI (le refus a bien bloqué l'action)
Contenu de corbeille/ : (vide)


### Scénario 6 — `repondre` : renvoyer une consigne sans exécuter

Variante subtile du refus : l'opérateur ne bloque pas définitivement, il **oriente**. Le modèle
reçoit la consigne et reprend la main.

C'est le patron *Review & Feedback* : il transforme la validation en dialogue, au lieu d'un
simple verrou binaire.

In [ ]:
demande = discuter(graphe, "Envoie un mail à direction@formatis.fr au sujet du projet Atlas", "atelier-5")


[UTILISATEUR] Envoie un mail à direction@formatis.fr au sujet du projet Atlas
   [appel outil] envoyer_email({"destinataire": "direction@formatis.fr", "sujet": "Projet Atlas", "corps": "Bonjour,\n\nFaisant suite à votre demande concernant projet Atl)
[AGENT] J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.

PAUSE — VALIDATION HUMAINE REQUISE
  Outil demandé : envoyer_email
  Arguments proposés par l'agent :
      - destinataire  : direction@formatis.fr
      - sujet         : Projet Atlas
      - corps         : Bonjour,
                        
                        Faisant suite à votre demande concernant projet Atlas, voici le message préparé automatiquement par l'agent.
                        
                        Bien cordialement,
                        Votre agent.
  Message de l'agent : J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.
  Décisions possibles :
      - approuver  : exécuter tel quel
      - modifier   :

In [ ]:
reprendre(graphe, {
    "action": "repondre",
    "validateur": "stephane",
    "message": "Trop direct pour la direction. Prépare d'abord une note interne, "
               "on enverra le mail après relecture.",
}, "atelier-5")


>>> DÉCISION HUMAINE : {"action": "repondre", "validateur": "stephane", "message": "Trop direct pour la direction. Prépare d'abord une note interne, on enverra le mail après relecture."}
   [journal] stephane a renvoyé une CONSIGNE sur envoyer_email
   [résultat outil : envoyer_email] Action non exécutée. Consigne de l'opérateur : Trop direct pour la direction. Prépare d'abord une note interne, on enverra le mail après relecture.…
[AGENT] Bien noté, je n'ai rien envoyé. Consigne retenue : Trop direct pour la direction. Prépare d'abord une note interne, on enverra le mail après relecture. Je peux préparer une nouvelle version sur cette base.


### Scénario 7 — Une chaîne complète : rechercher **puis** envoyer

La demande enchaîne deux outils de nature différente. Observez la trace :

1. `recherche_web` — lecture seule → **exécutée directement** ;
2. `envoyer_email` — effet de bord → **pause**, avec un brouillon déjà nourri par la recherche.

Un seul point d'arrêt, placé exactement là où il fallait.

In [ ]:
demande = discuter(graphe,
                   "Cherche ce qu'est la mémoire des agents conversationnels puis envoie un mail "
                   "de synthèse à leila@formatis.fr",
                   "atelier-6")


[UTILISATEUR] Cherche ce qu'est la mémoire des agents conversationnels puis envoie un mail de synthèse à leila@formatis.fr
   [appel outil] recherche_web({"requete": "ce qu'est la mémoire des agents conversationnels"})
[AGENT] Je lance une recherche documentaire.
   [résultat outil : recherche_web] [web/duckduckgo] Mémoire Conversationnelle | Tech Journey - Joseph He This document explores the crucial concept of conversational state, or memory, i…
   [appel outil] envoyer_email({"destinataire": "leila@formatis.fr", "sujet": "Synthèse : ce qu'est la mémoire des agents conversationnels", "corps": "Bonjour,\n\nVoici la)
[AGENT] Recherche terminée. Je prépare l'e-mail de synthèse : il attend votre validation.

PAUSE — VALIDATION HUMAINE REQUISE
  Outil demandé : envoyer_email
  Arguments proposés par l'agent :
      - destinataire  : leila@formatis.fr
      - sujet         : Synthèse : ce qu'est la mémoire des agents conversationnels
      - corps         : Bonjour,
                      

In [ ]:
reprendre(graphe, {"action": "approuver", "validateur": "stephane"}, "atelier-6")


>>> DÉCISION HUMAINE : {"action": "approuver", "validateur": "stephane"}
   [journal] stephane a APPROUVÉ envoyer_email
   [résultat outil : envoyer_email] E-mail envoyé à leila@formatis.fr — sujet « Synthèse : ce qu'est la mémoire des agents conversationnels » (672 caractères). Copie dans sortie/boite_en…
[AGENT] C'est fait : E-mail envoyé à leila@formatis.fr — sujet « Synthèse : ce qu'est la mémoire des agents conversationnels » (672 caractères). Copie dans sortie/boite_envoi.jsonl.


---
## Partie 9 — Persistance : la preuve par le redémarrage

Jusqu'ici, tout tenait en mémoire vive. Vérifions maintenant les deux promesses de la persistance.

**Promesse 1 — la mémoire longue traverse les conversations.** Un `thread_id` jamais utilisé :
l'historique est vide, mais l'agent sait toujours qui vous êtes.

In [ ]:
discuter(graphe, "Bonjour, sais-tu qui je suis ?", "conversation-toute-neuve")


[UTILISATEUR] Bonjour, sais-tu qui je suis ?
[AGENT] Bonjour ! Je suis votre agent conversationnel supervisé. Je garde en mémoire : L'utilisateur s'appelle Stéphane ; L'utilisateur travaille chez Formatis. Je peux chercher de l'information, lire des notes et préparer des e-mails — ces derniers passeront toujours par votre validation.


In [ ]:
etat = graphe.get_state({"configurable": {"thread_id": "conversation-toute-neuve"}})
print("Faits relus depuis SqliteStore :")
print(textwrap.indent(etat.values.get("contexte_memoire", "(aucun)"), "   "))

Faits relus depuis SqliteStore :
   - L'utilisateur s'appelle Stéphane
   - L'utilisateur travaille chez Formatis


**Promesse 2 — une validation en attente survit à un redémarrage.**

On simule un arrêt du serveur : nouvelles connexions SQLite, nouveau graphe compilé, aucun objet
Python en commun avec le précédent. La pause, elle, était sur disque.

C'est ce qui rend le patron utilisable en vrai : une demande de validation peut attendre qu'un
responsable la traite **le lendemain matin**, sans maintenir un processus vivant.

In [ ]:
# 1) un e-mail est proposé, puis on « perd » le processus
attente = discuter(graphe, "Envoie un mail à support@formatis.fr au sujet de l'incident de sauvegarde",
                   "atelier-longue-pause")
print("\n>>> Ici, on ferme le notebook, on éteint le serveur, on part en week-end…")


[UTILISATEUR] Envoie un mail à support@formatis.fr au sujet de l'incident de sauvegarde
   [appel outil] envoyer_email({"destinataire": "support@formatis.fr", "sujet": "Incident de sauvegarde", "corps": "Bonjour,\n\nFaisant suite à votre demande concernant in)
[AGENT] J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.

PAUSE — VALIDATION HUMAINE REQUISE
  Outil demandé : envoyer_email
  Arguments proposés par l'agent :
      - destinataire  : support@formatis.fr
      - sujet         : Incident de sauvegarde
      - corps         : Bonjour,
                        
                        Faisant suite à votre demande concernant incident de sauvegarde, voici le message préparé automatiquement par l'agent.
                        
                        Bien cordialement,
                        Votre agent.
  Message de l'agent : J'ai préparé un brouillon d'e-mail. Il attend votre validation avant envoi.
  Décisions possibles :
      - approuver  : exécuter t

In [ ]:
# 2) redémarrage complet : nouvelles connexions, nouveau graphe
del graphe
checkpointer_2, magasin_2 = ouvrir_memoires()
MAGASIN = magasin_2
graphe = construire_graphe(modele, checkpointer_2, magasin_2, ID_UTILISATEUR)

etat = graphe.get_state({"configurable": {"thread_id": "atelier-longue-pause"}})
print("Après redémarrage :")
print(f"  messages restaurés : {len(etat.values.get('messages', []))}")
print(f"  prochain nœud      : {', '.join(etat.next) if etat.next else '(fin de tour)'}")
print(f"  validation en attente : {'OUI' if etat.interrupts else 'non'}")
if etat.interrupts:
    charge = etat.interrupts[0].value
    print(f"  outil en attente   : {charge['outil_sensible']} → {charge['arguments']['destinataire']}")

Après redémarrage :
  messages restaurés : 2
  prochain nœud      : revision_humaine
  validation en attente : OUI
  outil en attente   : envoyer_email → support@formatis.fr


In [ ]:
# 3) le responsable traite la demande le lendemain, sur le graphe reconstruit
reprendre(graphe, {"action": "approuver", "validateur": "responsable_support"}, "atelier-longue-pause")


>>> DÉCISION HUMAINE : {"action": "approuver", "validateur": "responsable_support"}
   [journal] responsable_support a APPROUVÉ envoyer_email
   [résultat outil : envoyer_email] E-mail envoyé à support@formatis.fr — sujet « Incident de sauvegarde » (161 caractères). Copie dans sortie/boite_envoi.jsonl.…
[AGENT] C'est fait : E-mail envoyé à support@formatis.fr — sujet « Incident de sauvegarde » (161 caractères). Copie dans sortie/boite_envoi.jsonl.


---
## Partie 10 — Le journal d'audit

Une supervision humaine non tracée n'a aucune valeur probante. Chaque décision a été écrite dans
`sortie/journal_validations.jsonl` : **qui** a validé, **quoi**, **quand**, et avec quel résultat.

C'est ce fichier que réclamera un auditeur, dans le cadre de l'AI Act ou de l'article 22 du RGPD.

> **Passerelle vers le module 5** (*Perspectives et enjeux éthiques des agents IA*) : ce journal est
> la brique technique élémentaire de la gouvernance et de l'explicabilité (XAI) qui y seront
> traitées. Retenez que la traçabilité ne s'ajoute pas après coup — elle se conçoit en même temps
> que le point d'arrêt.

In [ ]:
print(f"{'ACTION':<12}{'OUTIL':<22}{'RÉSULTAT':<28}{'VALIDATEUR':<22}DATE")
print("-" * 104)
for ligne in (DOSSIERS["sortie"] / "journal_validations.jsonl").read_text().strip().splitlines():
    e = json.loads(ligne)
    print(f"{e['action']:<12}{e['outil']:<22}{e['resultat']:<28}{e['validateur']:<22}{e.get('date','')}")

lignes = (DOSSIERS["sortie"] / "journal_validations.jsonl").read_text().strip().splitlines()
executees = sum(1 for l in lignes if json.loads(l)["resultat"].startswith("execute"))
print(f"\n{len(lignes)} décisions humaines — {executees} actions exécutées, "
      f"{len(lignes) - executees} bloquées avant effet.")

ACTION      OUTIL                 RÉSULTAT                    VALIDATEUR            DATE
--------------------------------------------------------------------------------------------------------
approuver   envoyer_email         execute                     stephane              2026-09-20T21:35:03+00:00
modifier    envoyer_email         execute_apres_correction    stephane              2026-09-20T21:35:38+00:00
refuser     supprimer_document    refuse                      stephane              2026-09-20T21:37:50+00:00
repondre    envoyer_email         consigne                    stephane              2026-09-20T21:38:41+00:00
approuver   envoyer_email         execute                     stephane              2026-09-20T21:39:38+00:00
approuver   envoyer_email         execute                     responsable_support   2026-09-20T21:41:07+00:00

6 décisions humaines — 4 actions exécutées, 2 bloquées avant effet.


---
## Partie 11 — Mode interactif (facultatif)

Pour éprouver l'agent librement. Passez `MODE_INTERACTIF = True` puis exécutez la cellule :
dialoguez, et répondez aux demandes de validation par `approuver`, `refuser`, `modifier`
ou `repondre`. Tapez `quitter` pour sortir.

> Laissé à `False` par défaut : une cellule qui attend une saisie bloque un
> « Exécuter tout ».

Quelques demandes à essayer :

- `Je préfère les réponses courtes et structurées.` → mémorisation d'une préférence
- `Lis la note sur la réunion Atlas` → outil de lecture, sans pause
- `Supprime le fichier export_temporaire.txt` → pause, puis approuvez
- `Envoie un mail à qui-de-droit@formatis.fr au sujet du planning` → pause, puis essayez `modifier`

In [ ]:
MODE_INTERACTIF = False        # <<< passez à True pour dialoguer librement

if MODE_INTERACTIF:
    fil = "session-interactive"
    print("Agent prêt. Tapez « quitter » pour sortir.\n")
    while True:
        saisie = input("Vous > ").strip()
        if saisie.lower() in {"quitter", "exit", "q", ""}:
            print("Fin de session.")
            break
        attente = discuter(graphe, saisie, fil)
        while attente:
            choix = input("Décision [approuver/modifier/refuser/repondre] > ").strip().lower()
            if choix == "modifier":
                champ = input("  champ à corriger > ").strip()
                valeur = input("  nouvelle valeur > ")
                decision = {"action": "modifier", "arguments": {champ: valeur},
                            "validateur": "operateur"}
            elif choix == "refuser":
                decision = {"action": "refuser", "motif": input("  motif > ").strip(),
                            "validateur": "operateur"}
            elif choix == "repondre":
                decision = {"action": "repondre", "message": input("  consigne > ").strip(),
                            "validateur": "operateur"}
            else:
                decision = {"action": "approuver", "validateur": "operateur"}
            attente = reprendre(graphe, decision, fil)
else:
    print("MODE_INTERACTIF = False → cellule ignorée (passez-le à True pour dialoguer).")

MODE_INTERACTIF = False → cellule ignorée (passez-le à True pour dialoguer).


---
## Partie 12 — Variante industrielle : `HumanInTheLoopMiddleware`

Nous avons construit le point d'arrêt **à la main**, ce qui était l'objectif pédagogique : vous
savez maintenant ce qui se passe réellement. En production, LangChain 1.x fournit le même patron
clé en main, via `create_agent` et un *middleware*.

| | Graphe explicite (parties 6-8) | `HumanInTheLoopMiddleware` |
|---|---|---|
| Code à écrire | ~70 lignes | ~8 lignes |
| Contrôle du format d'interruption | total | imposé |
| Aiguillage sur mesure (par argument, par montant…) | oui | limité à la configuration |
| Recommandé pour | cas métier spécifiques, apprentissage | mise en place rapide et standard |

Attention aux formats, qui diffèrent de notre implémentation :

- la charge utile contient `action_requests` (une **liste**) et `review_configs` ;
- la reprise attend `Command(resume={"decisions": [{"type": "approve"}]})` ;
- les types de décision sont `approve`, `edit` (avec `edited_action`) et `reject` (avec `message`).

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent_middleware = create_agent(
    model=modele,
    tools=[recherche_web, lire_note, envoyer_email, supprimer_document],
    system_prompt=INVITE_SYSTEME,
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "envoyer_email":      {"allowed_decisions": ["approve", "edit", "reject"]},
            "supprimer_document": {"allowed_decisions": ["approve", "reject"]},
            "recherche_web": False,    # lecture seule : jamais d'interruption
            "lire_note": False,
        },
        # le middleware insère une note à destination du modèle après une correction ;
        # elle est en anglais par défaut, on la traduit :
        edit_notice="Note : un relecteur humain a remplacé cet appel d'outil avant son "
                    "exécution. C'était intentionnel et autorisé ; ne réémets pas ton appel initial.",
    )],
    checkpointer=InMemorySaver(),   # InMemorySaver suffit pour cette démonstration
)

config_mw = {"configurable": {"thread_id": "middleware-1"}}
sortie = agent_middleware.invoke(
    {"messages": [HumanMessage(content="Envoie un mail à audit@formatis.fr au sujet du projet Atlas")]},
    config_mw)

demande_mw = agent_middleware.get_state(config_mw).interrupts[0].value
print("Interruption produite par le middleware :\n")
for requete in demande_mw["action_requests"]:
    print(f"  outil     : {requete['name']}")
    print(f"  arguments : {json.dumps(requete['args'], ensure_ascii=False)[:200]}…")
for configuration in demande_mw["review_configs"]:
    print(f"  décisions autorisées pour {configuration['action_name']} : {configuration['allowed_decisions']}")

Interruption produite par le middleware :

  outil     : envoyer_email
  arguments : {"destinataire": "audit@formatis.fr", "sujet": "Projet Atlas", "corps": "Bonjour,\n\nFaisant suite à votre demande concernant projet Atlas, voici le message préparé automatiquement par l'agent.\n\nBie…
  décisions autorisées pour envoyer_email : ['approve', 'edit', 'reject']


In [ ]:
# Reprise au format du middleware — ici une correction (edit)
sortie = agent_middleware.invoke(
    Command(resume={"decisions": [{
        "type": "edit",
        "edited_action": {
            "name": "envoyer_email",
            "args": {"destinataire": "audit-interne@formatis.fr",
                     "sujet": "Projet Atlas — point d'étape",
                     "corps": "Bonjour,\n\nCorrigé par l'opérateur avant envoi.\n\nStéphane"},
        },
    }]}),
    config_mw)

print("Réponse finale de l'agent :", texte_de(sortie["messages"][-1])[:200])
print("\nDernier e-mail de la boîte d'envoi :")
print(textwrap.indent(json.dumps(
    json.loads((DOSSIERS["sortie"] / "boite_envoi.jsonl").read_text().strip().splitlines()[-1]),
    ensure_ascii=False, indent=2), "   "))

Réponse finale de l'agent : C'est fait : Note : un relecteur humain a remplacé cet appel d'outil avant son exécution. C'était intentionnel et autorisé ; ne réémets pas ton appel initial. Executed instead: envoyer_email with argu

Dernier e-mail de la boîte d'envoi :
   {
     "date": "2026-09-20T21:43:02+00:00",
     "destinataire": "audit-interne@formatis.fr",
     "sujet": "Projet Atlas — point d'étape",
     "corps": "Bonjour,\n\nCorrigé par l'opérateur avant envoi.\n\nStéphane"
   }


---
## Partie 13 — Exercices

Quatre exercices de difficulté croissante. Les corrigés suivent : essayez d'abord.

### Exercice 1 — Un seuil métier *(facile)*
Ajoutez un outil `virement(beneficiaire: str, montant: float)`. Il ne doit déclencher une
validation humaine **que si `montant > 500`**. En dessous, exécution directe.
*Indice : la fonction `aiguiller` ne doit plus regarder seulement le nom de l'outil, mais aussi
ses arguments.*

### Exercice 2 — Le délai d'expiration *(moyen)*
Une validation qui n'arrive jamais bloque le dossier. Écrivez une fonction
`reprendre_avec_delai(graphe, thread, secondes)` qui, passé le délai, reprend automatiquement
l'exécution avec `{"action": "refuser", "motif": "délai de validation dépassé"}`.
*Indice : l'état en pause est lisible par `graphe.get_state(...)`; comparez l'horodatage du
checkpoint à l'heure courante.*

### Exercice 3 — Supprimer la variable globale *(moyen)*
Les outils utilisent la globale `MAGASIN`, ce qui interdit de servir deux utilisateurs
simultanément. Faites passer l'identifiant de l'utilisateur par la configuration du graphe.
*Indice : `from langgraph.config import get_config, get_store` — utilisables à l'intérieur d'un
outil exécuté dans le graphe.*

### Exercice 4 — Deux niveaux d'habilitation *(avancé)*
Distinguez deux rôles : un **opérateur** peut approuver un e-mail, seul un **responsable** peut
approuver une suppression. Si le validateur n'a pas le niveau requis, la décision est convertie en
refus motivé, et journalisée comme telle.
*Indice : l'objet `decision` rendu par `interrupt()` contient déjà `validateur` ; ajoutez-y un
champ `role` et une table d'habilitations.*

In [ ]:
# ---- Votre espace de travail pour les exercices ----
# (le corrigé de l'exercice 1 se trouve dans la cellule suivante)

### Corrigé de l'exercice 1 — seuil métier

Le point important : la règle d'aiguillage n'appartient pas au modèle, elle appartient au
**code de l'application**. Un LLM ne doit jamais décider lui-même s'il a besoin d'une autorisation.

In [ ]:
SEUIL_VALIDATION_EUROS = 500.0


@tool
def virement(beneficiaire: str, montant: float) -> str:
    """Effectue un virement bancaire. Au-delà de 500 €, validation humaine obligatoire."""
    _journaliser("boite_envoi.jsonl", {"date": _horodatage(), "type": "virement",
                                       "beneficiaire": beneficiaire, "montant": montant})
    return f"Virement de {montant:.2f} € exécuté vers {beneficiaire}."


def appel_necessite_validation(appel: dict) -> bool:
    """Règle métier : le NOM de l'outil, mais aussi la VALEUR de ses arguments."""
    if appel["name"] == "virement":
        return float(appel["args"].get("montant", 0)) > SEUIL_VALIDATION_EUROS
    return appel["name"] in OUTILS_SENSIBLES


for essai in [
    {"name": "virement", "args": {"beneficiaire": "Fournisseur A", "montant": 120.0}},
    {"name": "virement", "args": {"beneficiaire": "Fournisseur B", "montant": 4300.0}},
    {"name": "recherche_web", "args": {"requete": "test"}},
    {"name": "envoyer_email", "args": {"destinataire": "x@y.fr", "sujet": "s", "corps": "c"}},
]:
    verdict = "VALIDATION HUMAINE" if appel_necessite_validation(essai) else "exécution directe"
    detail = essai["args"].get("montant", "")
    print(f"  {essai['name']:<18}{str(detail):>10}  →  {verdict}")

print("\nPour l'intégrer : remplacez, dans `aiguiller`, le test "
      "`a['name'] in OUTILS_SENSIBLES` par `appel_necessite_validation(a)`.")

### Corrigé de l'exercice 2 — délai d'expiration

Une validation en attente possède un horodatage de checkpoint. Il suffit de le comparer à
l'heure courante ; passé le délai, on reprend l'exécution avec un refus automatique.
En production, ce contrôle est confié à une tâche planifiée qui balaie les threads en pause.

In [ ]:
from datetime import datetime, timedelta


def reprendre_avec_delai(graphe, thread: str, secondes: int = 3600, afficher: bool = True):
    """Refuse automatiquement une validation qui dépasse le délai imparti."""
    config = {"configurable": {"thread_id": thread}}
    etat = graphe.get_state(config)
    if not etat.interrupts:
        return None                                   # rien en attente

    depuis = datetime.fromisoformat(etat.created_at.replace("Z", "+00:00"))
    age = datetime.now(depuis.tzinfo) - depuis
    if afficher:
        print(f"Validation en attente depuis {age.total_seconds():.1f} s (délai : {secondes} s)")
    if age > timedelta(seconds=secondes):
        if afficher:
            print("→ délai dépassé : refus automatique.")
        return reprendre(graphe, {"action": "refuser", "validateur": "systeme",
                                  "motif": "délai de validation dépassé"}, thread, afficher)
    if afficher:
        print("→ dans les temps : on continue d'attendre.")
    return etat.interrupts[0].value


# démonstration : une demande toute fraîche avec un délai très court, puis un délai long
demande = discuter(graphe, "Supprime le fichier export_temporaire.txt", "atelier-delai", afficher=False)
print("Demande en attente :", demande["outil_sensible"], demande["arguments"])
print()
reprendre_avec_delai(graphe, "atelier-delai", secondes=3600)   # largement dans les temps
print()
reprendre_avec_delai(graphe, "atelier-delai", secondes=0)      # expiration immédiate
print()
print("export_temporaire.txt encore présent :",
      (DOSSIERS["documents"] / "export_temporaire.txt").exists())

---
## Pour aller plus loin

**Ce qu'il faut retenir de cet atelier**

1. Un LLM ne fait **jamais** qu'émettre une intention (`tool_call`) ; l'exécution appartient à
   votre code. Le contrôle humain s'insère dans cet interstice.
2. `interrupt()` fige l'état dans le checkpointer et rend la main ; `Command(resume=...)` reprend.
   **Le nœud est rejoué depuis le début** : aucun effet de bord avant l'appel à `interrupt()`.
3. Classer les outils par risque est une décision d'architecture, pas un réglage. Faire tout
   valider revient à ne rien faire valider (*rubber-stamping*).
4. Les deux mémoires ont des rôles distincts : checkpointer (une conversation) et store (un
   utilisateur). Le premier conditionne l'existence même du HITL.
5. Sans journal d'audit, une validation humaine n'est pas opposable.

**Pistes d'approfondissement**

- Remplacer SQLite par `langgraph-checkpoint-postgres` pour un déploiement multi-instances.
- Exposer les demandes de validation dans une interface (file d'attente, notification, e-mail).
- Explorer le *time-travel* : `graphe.get_state_history(config)` permet de rejouer une
  conversation depuis n'importe quel point.
- Ajouter une validation par lots : regrouper plusieurs actions en une seule demande.

**Ce que la suite de la formation va en faire**

| Module | Question que ce patron y soulève |
|---|---|
| **2** — Patrons d'architectures multi-agents | Dans une architecture supervisée, **qui** valide : l'humain, ou l'agent superviseur ? Où place-t-on le point d'arrêt quand un superviseur délègue à des agents spécialisés ? |
| **3** — Orchestration et planification | Fait-on valider **le plan** une fois, ou **chaque étape** de son exécution ? Le *Plan-and-Execute* déplace naturellement le point d'arrêt vers le plan. |
| **4** — Systèmes réflexifs et critiques | L'auto-évaluation d'un agent réflexif **remplace-t-elle** la validation humaine, ou la **complète-t-elle** ? (réponse courte : elle la filtre, elle ne la remplace pas) |
| **5** — Perspectives et enjeux éthiques | Le journal d'audit de la partie 10 devient la pièce maîtresse de la conformité AI Act. |

**Nettoyage (facultatif)** — la cellule ci-dessous efface les mémoires SQLite pour repartir de zéro.

In [ ]:
NETTOYER = False        # <<< passez à True pour effacer les mémoires et recommencer

if NETTOYER:
    import shutil as _shutil
    _shutil.rmtree(RACINE, ignore_errors=True)
    print(f"{RACINE} supprimé. Réexécutez le notebook depuis la partie 1.")
else:
    tailles = {f.name: f.stat().st_size for f in DOSSIERS["memoire"].glob("*.sqlite")}
    print("Mémoires conservées :", json.dumps(tailles, ensure_ascii=False))
    print("Passez NETTOYER = True pour repartir de zéro.")

---

*Atelier IA102 — Human-in-the-Loop avec LangGraph.
Versions validées le 20 septembre 2026 : langgraph 1.2.11 · langchain 1.4.2 · langchain-core 1.6.3
· langgraph-checkpoint-sqlite 3.1.1.*